# Kenya Land Valuation: Modelling (Notebook 3)

This notebook is **Notebook 3 of 3** in the Kenya land price project. It loads the engineered
dataset produced by Notebook 1 (`01_data_preparation_and_feature_engineering.ipynb`) —
`data/interim/kenya_land_engineered_final.parquet` — and trains regression models that predict
a parcel's **log price per acre** (`log_price_per_acre`, the natural log of KSh per acre).

**Goal.** Compare an OLS baseline with XGBoost, CatBoost and TabNet, combine the three
learners into a weighted ensemble, find the best feature set with a systematic evaluation, and
report **Run A** (the original 9 features used by the archived model section) against
**Run B** (the feature set selected in Phase 3) — plus **Run C** (the minimum-CV-error set
from the same search) when `CONFIG.RUN_ALTERNATIVE` is True. Every decision —
hyperparameters, features, loss functions, ensemble weights, which run becomes production — is
made with cross-validation on the development set only. The test set is read once per run, in
the final evaluation cell, and is never used to choose anything.

**Rows and missing values.** The modelling frame is every parquet row that passes the archived
target filters (`size_acres > 0`, `price_per_acre > 0`, finite target); the count is printed at
run time in Cell 1.8. The archived model section also dropped rows with nulls in the 9 base
features, which — because `dist_to_water_body_km` is missing **by design** for counties that
do not border a water body — would leave only a handful of rows; that step is therefore not
applied here. XGBoost and CatBoost handle the remaining NaNs natively; OLS and TabNet use
training-median imputation plus missing-indicator columns; and a `has_water_body_distance`
indicator joins the Phase 3 candidate pool.

**Inputs.**

- `data/interim/kenya_land_engineered_final.parquet` — the final engineered dataset from
  Notebook 1. This is the **only** data file the notebook reads; it depends on no variable
  from any other notebook.

**Outputs.**

- Backend artefacts in `models/ensemble/`: `xgb_model.json`, `catboost_model.cbm`,
  `tabnet_model.zip`, `ensemble_weights.json`, `feature_list.json`, `preprocessing.json`,
  `metadata.json` (Phase 8).
- Interpretation charts in `outputs/modelling/`: `pred_vs_actual.png`, `residuals.png`,
  `model_comparison.png`, `feature_importance.png`, `shap_summary.png` (Phase 7).
- Resume artefacts per run in `models/runs/<run>/`: Optuna studies (`optuna_*.db`, opened
  with `load_if_exists=True`), out-of-fold predictions (`oof_*.json`) and the
  development-fitted models (`refit_*` files, `ensemble_weights.json`, `preprocessing.json`).
  Finished stages are skipped on re-runs; `CONFIG.FORCE_RERUN` recomputes them.

**How to run.** Run the cells in order on Google Colab with Google Drive mounted
(Cell 1.1 installs the extra libraries; Cell 1.3 mounts Drive; Cell 1.4 defines the same
paths as Notebook 1). Edit the CONFIG cell (1.6) to change the cross-validation and tuning
budgets, or set `FAST_DEBUG = True` for a quick end-to-end mechanics check. Every run uses
identical tuning budgets and identical cross-validation folds, sized so a full run needs
about three hours or less on a CPU runtime. Interrupted runs resume: a stage whose saved files
already exist is skipped. This notebook never writes inside Notebook 1's data folders; it only
adds files under `models/` and `outputs/modelling/`.


<a id="contents"></a>

## Contents

**Sections**

- [Handoff from Notebook 1](#handoff)
- [Phase 1 — Setup and Data](#phase-1)
- [Phase 2 — Baselines](#phase-2)
- [Phase 3 — Feature Evaluation](#phase-3)
- [Phase 4 — Base Models](#phase-4)
- [Phase 5 — Ensemble](#phase-5)
- [Phase 6 — Evaluation](#phase-6)
- [Phase 7 — Interpretation](#phase-7)
- [Phase 8 — Save for Backend](#phase-8)

**Cells**

- [Cell 1.1 — Install Dependencies](#cell-1-1) · [Cell 1.2 — Imports and Versions](#cell-1-2) · [Cell 1.3 — Mount Google Drive](#cell-1-3) · [Cell 1.4 — Paths](#cell-1-4) · [Cell 1.5 — Seeds and Device](#cell-1-5) · [Cell 1.6 — CONFIG](#cell-1-6) · [Cell 1.7 — Load Data and Check Columns](#cell-1-7) · [Cell 1.8 — Filter and Split](#cell-1-8) · [Cell 1.9 — Data-Quality Snapshot](#cell-1-9) · [Cell 1.10 — Metrics Helper](#cell-1-10)
- [Cell 2.1 — OLS Baseline](#cell-2-1) · [Cell 2.2 — Training-Median Baseline](#cell-2-2)
- [Cell 3.1 — Candidate Pool and Availability](#cell-3-1) · [Cell 3.2 — Preprocessing and Fold Plan](#cell-3-2) · [Cell 3.3 — Pre-Screen](#cell-3-3) · [Cell 3.4 — Group Ablation](#cell-3-4) · [Cell 3.5 — Permutation Importance](#cell-3-5) · [Cell 3.6 — SHAP Screening](#cell-3-6) · [Cell 3.7 — Greedy Selection and Stability](#cell-3-7) · [Cell 3.8 — Ranked Table, SELECTED and ALTERNATIVE Features](#cell-3-8)
- [Cell 4.1 — County Group Folds](#cell-4-1) · [Cell 4.2 — XGBoost Tuning](#cell-4-2) · [Cell 4.3 — CatBoost Tuning](#cell-4-3) · [Cell 4.4 — TabNet Tuning](#cell-4-4)
- [Cell 5.1 — Ensemble Weights](#cell-5-1) · [Cell 5.2 — Refit on Development Data](#cell-5-2)
- [Cell 6.1 — Test Evaluation Helpers](#cell-6-1) · [Cell 6.2 — Sensitivity and County CV](#cell-6-2) · [Cell 6.3 — Reusable Run Routine](#cell-6-3) · [Cell 6.4 — Run A](#cell-6-4) · [Cell 6.5 — Run B and Optional Run C](#cell-6-5) · [Cell 6.6 — Comparison Table](#cell-6-6)
- [Cell 7.1 — Predicted vs Actual and Residuals](#cell-7-1) · [Cell 7.2 — Model Comparison Chart](#cell-7-2) · [Cell 7.3 — Feature Importance](#cell-7-3) · [Cell 7.4 — SHAP Summary](#cell-7-4)
- [Cell 8.1 — Production Run Selection](#cell-8-1) · [Cell 8.2 — Save Artefacts](#cell-8-2) · [Cell 8.3 — predict_price_per_acre](#cell-8-3)


<a id="handoff"></a>

## Handoff from Notebook 1

Notebook 1 ends at Cell 8.3 by saving the engineered dataset to
`data/interim/kenya_land_engineered_final.parquet` (CSV copy in `data/raw/`). This notebook
reads that parquet and nothing else. Expected columns — the 28 `FINAL_COLUMNS` of Notebook 1,
in order:

`listing_url`, `listing_no`, `title`, `location`, `prop_type`, `list_date`, `price_ksh`,
`price_flag`, `log_price_ksh`, `size_acres`, `size_source`, `log_size_acres`, `latitude`,
`longitude`, `geocode_source`, `geocode_confidence`, `county`, `dist_to_nairobi_km`,
`dist_to_county_town_km`, `dist_to_water_body_km`, `reference_city`, `reference_city_dist_km`,
`amenities_score`, `accessibility_score`, `infrastructure_score`, `infra_source`,
`price_per_acre`, `log_price_per_acre`

How this notebook uses them:

| Role | Columns |
|---|---|
| Target | `log_price_per_acre` |
| Run A features (the 9 base features of the archived model section) | `log_size_acres`, `dist_to_nairobi_km`, `dist_to_county_town_km`, `dist_to_water_body_km`, `reference_city_dist_km`, `geocode_confidence`, `amenities_score`, `accessibility_score`, `infrastructure_score` |
| Additional Phase 3 candidates | `latitude`, `longitude`, `prop_type`, `county`, `reference_city`, `geocode_source`, `infra_source`, `size_source`, the derived `has_water_body_distance` (Cell 1.8), and `listing_year` / `listing_month` derived from `list_date` |
| Row filtering (archived Cell M1's target filters) | `size_acres`, `price_per_acre`, `log_price_per_acre` |
| Never used as features | `price_ksh`, `log_price_ksh`, `price_per_acre`, `price_flag` (price family and the target itself), `listing_url`, `listing_no`, `title`, `location` (identifiers and free text) |

Cell 1.7 fails with a clear message if any required column is missing. All paths are relative
to `PROJECT_ROOT = /content/drive/MyDrive/kenya_land_valuation` (Cell 1.4), exactly as in
Notebook 1 and the archived model section.


<a id="phase-1"></a>

## Phase 1 — Setup and Data

Installs the modelling libraries, mounts Google Drive, fixes seeds and the device, exposes the
single editable CONFIG cell, loads the engineered parquet, applies the archived model
section's target filters (`size_acres > 0`, `price_per_acre > 0`, finite target — **no**
null-drop on the features; see Cell 1.8) and its two `train_test_split` calls, prints a
data-quality snapshot of the resulting frame (Cell 1.9), and ends with the metric helpers
used throughout the notebook. Every row count is computed and printed at run time.


<a id="cell-1-1"></a>

### Cell 1.1 — Install Dependencies

`xgboost` and `shap` ship with recent Colab images; `pip` treats them as satisfied no-ops then.
`catboost`, `pytorch-tabnet` and `optuna` are the additions this notebook needs.


In [ ]:
# Cell 1.1 — Install the libraries that a stock Colab runtime lacks.
!pip -q install xgboost catboost pytorch-tabnet optuna shap
print("PHASE 1.1 — dependencies installed (xgboost, catboost, pytorch-tabnet, optuna, shap).")

<a id="cell-1-2"></a>

### Cell 1.2 — Imports and Versions

Every library used anywhere in the notebook is imported here, once, and the versions are
printed so saved logs stay interpretable. The phase timer defined at the bottom is started at
the top of every phase and printed at its end, so each phase's elapsed time lands in the log.


In [ ]:
# Cell 1.2 — Imports (single place) and library versions.
import hashlib
import json
import os
import platform
import random
import sys
import time
import warnings
from importlib.metadata import version as _pkg_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import minimize

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, RepeatedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
import shap
import torch
from pytorch_tabnet.tab_model import TabNetRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Per-phase elapsed-time reporting: a phase's first code cell calls phase_timer_start(),
# its last code cell calls phase_timer_print(...).
PHASE_T0 = time.time()

def phase_timer_start():
    global PHASE_T0
    PHASE_T0 = time.time()

def phase_timer_print(label):
    dt = time.time() - PHASE_T0
    print(f"[elapsed] {label}: {dt:.0f}s ({dt / 60:.1f} min)")

print("PHASE 1.2 — library versions:")
for _p in ["numpy", "pandas", "scipy", "scikit-learn", "xgboost", "catboost",
           "optuna", "shap", "torch", "pytorch-tabnet", "matplotlib"]:
    print(f"  {_p:14s} {_pkg_version(_p)}")
print(f"  {'python':14s} {platform.python_version()}")

<a id="cell-1-3"></a>

### Cell 1.3 — Mount Google Drive

Colab-only. When running outside Colab the mount is skipped and the paths of Cell 1.4 must
already resolve on the local machine.


In [ ]:
# Cell 1.3 — Mount Google Drive (Colab); skip gracefully elsewhere.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("PHASE 1.3 — Google Drive mounted.")
except ImportError:
    print("PHASE 1.3 — not running inside Colab; skipping the Drive mount.")

<a id="cell-1-4"></a>

### Cell 1.4 — Paths

Same variable names and values as Notebook 1 (Cell 0.2) and the archived model section
(Cell M0): `PROJECT_ROOT`, `INTERIM_DATA`, `MODEL_DIR`, `OUTPUTS`; `RUN_DIR` adds a per-run
folder for the resume artefacts (Optuna studies, out-of-fold predictions, development-fitted
models). The only data input of this notebook is `DATA_PATH`.


In [ ]:
# Cell 1.4 — Project paths (names and values identical to Notebook 1 / archived M0).
PROJECT_ROOT = "/content/drive/MyDrive/kenya_land_valuation"
INTERIM_DATA = f"{PROJECT_ROOT}/data/interim"
MODEL_DIR    = f"{PROJECT_ROOT}/models"
OUTPUTS      = f"{PROJECT_ROOT}/outputs/modelling"
RUN_DIR      = f"{MODEL_DIR}/runs"   # per-run resume artefacts: studies, OOF, refits

# This notebook's only data input.
DATA_PATH = f"{INTERIM_DATA}/kenya_land_engineered_final.parquet"

for _d in (MODEL_DIR, OUTPUTS, RUN_DIR):
    os.makedirs(_d, exist_ok=True)

print("PHASE 1.4 — paths:")
print(f"  PROJECT_ROOT : {PROJECT_ROOT}")
print(f"  INTERIM_DATA : {INTERIM_DATA}")
print(f"  MODEL_DIR    : {MODEL_DIR}")
print(f"  OUTPUTS      : {OUTPUTS}")
print(f"  RUN_DIR      : {RUN_DIR}")
print(f"  DATA_PATH    : {DATA_PATH}")

<a id="cell-1-5"></a>

### Cell 1.5 — Seeds and Device

`SEED = 42` matches the split seed of the archived model section. Every stochastic component
(additional splits, Optuna samplers, TabNet, bootstrap) is re-seeded explicitly with this value
as well, so a full re-run reproduces itself.


In [ ]:
# Cell 1.5 — Global seed and device.
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PHASE 1.5 — global setup:")
print(f"  SEED  : {SEED}")
print(f"  DEVICE: {DEVICE} (TabNet uses the GPU automatically when one is available)")

<a id="cell-1-6"></a>

### Cell 1.6 — CONFIG

The single editable control panel. `N_FOLDS x CV_REPEATS` repeated-K-fold cross-validation is
the decision engine of the notebook. The `N_TRIALS_*` budgets are Optuna search sizes per model
per run — identical for every run — and are chosen so a full run needs about three hours or
less on a CPU runtime. `FAST_DEBUG` shrinks every budget for a quick end-to-end mechanics
check (results from a FAST_DEBUG pass must never be used for decisions). `FORCE_RERUN`
recomputes stages whose saved files already exist (see below). `RUN_ALTERNATIVE` additionally
runs the minimum-CV-error feature set from Phase 3 as **Run C**. `SELECTED_FEATURES` is
filled automatically by Cell 3.8; hard-code a list here instead to skip the Phase 3 search.

**Resume support.** Optuna studies live in SQLite files under `models/runs/<run>/` and are
opened with `load_if_exists=True`; out-of-fold predictions and the development-fitted models
are saved per run. A stage whose files already exist (and match the current configuration
signature) is skipped, so an interrupted run resumes where it stopped — set
`FORCE_RERUN = True` to recompute everything.


In [ ]:
# Cell 1.6 — CONFIG (the single editable control panel).
N_FOLDS           = 5      # K-fold splits inside each CV repeat
CV_REPEATS        = 2      # number of CV repeats (stability of every estimate)
N_TRIALS_XGB      = 30     # Optuna budget per run, XGBoost
N_TRIALS_CATBOOST = 20     # Optuna budget per run, CatBoost
N_TRIALS_TABNET   = 8      # Optuna budget per run, TabNet (the slow model)
FAST_DEBUG        = False  # True -> tiny budgets, mechanics check only
FORCE_RERUN       = False  # True -> ignore finished stage files and recompute them
RUN_ALTERNATIVE   = False  # True -> also run ALTERNATIVE_FEATURES as Run C (Cell 6.5)
SELECTED_FEATURES = []     # filled by Cell 3.8; hard-code here to override Phase 3

# The budgets above target a full run of roughly three hours or less on a CPU runtime.
# Stage results persist under models/runs/<run>/ (Optuna SQLite studies, out-of-fold
# predictions, development-fitted models); a finished stage is skipped when its files
# already exist, so interrupted runs resume where they stopped. FORCE_RERUN = True
# recomputes every stage from scratch.

if FAST_DEBUG:
    N_FOLDS, CV_REPEATS = 3, 2
    N_TRIALS_XGB, N_TRIALS_CATBOOST, N_TRIALS_TABNET = 3, 3, 2
    print("!!! FAST_DEBUG is ON — budgets shrunk; use this pass to check mechanics only.")

print(f"PHASE 1.6 — CONFIG: folds={N_FOLDS}, repeats={CV_REPEATS}, "
      f"trials (xgb/cat/tab) = {N_TRIALS_XGB}/{N_TRIALS_CATBOOST}/{N_TRIALS_TABNET}, "
      f"FAST_DEBUG={FAST_DEBUG}, FORCE_RERUN={FORCE_RERUN}, RUN_ALTERNATIVE={RUN_ALTERNATIVE}")

### About the archived OLS / MLP reference numbers (kept as a note, not compared)

The archived model section additionally dropped every row with a null in the 9 base
features, which — because `dist_to_water_body_km` is missing by design for counties that do
not border a water body — left only a 32-row modelling frame: **22 train / 5 validation /
5 test rows**. This notebook models on the much larger target-filtered frame (counted at run
time in Cell 1.8), so the archived test metrics below are **not comparable** with the runs
here and are deliberately excluded from the Phase 6 comparison table. They are kept only as
the historical record:

| Archived model (printed by Cells M5 / M8) | R2 | RMSE | MAE |
|---|---|---|---|
| OLS | 0.7123 | 1.3606 | 1.1784 |
| MLP ("invalid setup") | -8.6094 | 7.8633 | 7.7651 |

The MLP row deserves its label: the archived section trained the network on the **unscaled**
`log_price_per_acre` target, so it converged to a near-constant prediction offset — its
residual summary reports a residual mean of 7.7651 with a standard deviation of only 1.2389,
i.e. an almost constant bias. That is a setup failure, not a property of neural networks on
this dataset. The old `REFERENCE_RESULTS` constant that used to carry these numbers as
comparison-table rows has been removed; TabNet in this notebook trains on a target
standardised with training-fold statistics and inverts the standardisation on its predictions,
precisely to avoid this failure mode.


<a id="cell-1-7"></a>

### Cell 1.7 — Load Data and Check Columns

Reads the parquet and verifies that **every** column this notebook touches exists, failing with
a clear message otherwise. No other file is ever read.


In [ ]:
# Cell 1.7 — Load the engineered parquet and check every required column.
TARGET = "log_price_per_acre"

BASE_FEATURES = [
    # Size
    "log_size_acres",
    # Proximity
    "dist_to_nairobi_km",
    "dist_to_county_town_km",
    "dist_to_water_body_km",
    "reference_city_dist_km",
    # Geocode quality
    "geocode_confidence",
    # Pillar scores
    "amenities_score",
    "accessibility_score",
    "infrastructure_score",
]

# Everything this notebook reads: features, target, filtering and candidate columns.
REQUIRED_COLUMNS = sorted(set(
    BASE_FEATURES
    + [TARGET]
    + ["size_acres", "price_per_acre", "price_ksh", "log_price_ksh", "price_flag", "list_date"]
    + ["latitude", "longitude", "prop_type", "county", "reference_city"]
    + ["geocode_source", "infra_source", "size_source"]
))

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Data file not found: {DATA_PATH}\n"
        "Run Notebook 1 (01_data_preparation_and_feature_engineering.ipynb) first, "
        "or check the Drive mount (Cell 1.3) and the paths (Cell 1.4)."
    )

df_full = pd.read_parquet(DATA_PATH)
print(f"PHASE 1.7 — raw records loaded : {len(df_full):,}")
print(f"          columns in parquet  : {len(df_full.columns)}")

_missing = [c for c in REQUIRED_COLUMNS if c not in df_full.columns]
if _missing:
    raise ValueError(
        f"The parquet is missing required columns: {_missing}\n"
        f"Expected Notebook 1's 28 FINAL_COLUMNS. Columns found: {list(df_full.columns)}"
    )
print("          all required columns present.")

<a id="cell-1-8"></a>

### Cell 1.8 — Filter and Split

Applies the archived model section's target filters (Cell M1) — drop rows with `size_acres <= 0`,
`price_per_acre <= 0` or a non-finite target — and keeps the archived split calls (Cell M3)
exactly: `test_size=0.15, random_state=42`, then `test_size=0.176, random_state=42` on the
remainder. The archived null-drop on the 9 base features (Cell M2) is deliberately **not**
applied: `dist_to_water_body_km` is missing **by design** for counties that do not border a
water body, so requiring it non-null would shrink the frame to a handful of rows. Missing
values are handled per model instead — native NaN handling for XGBoost and CatBoost,
training-median imputation plus missing-indicator columns for OLS and TabNet — and Cell 1.9
prints the missingness the models will see.

If fewer than 300 modelling rows survive the target filters, the cell stops with a clear
message — modelling on fewer rows is not meaningful. `has_water_body_distance` (1 when
`dist_to_water_body_km` is present, else 0) is derived here and joins the Phase 3 candidate
pool. Counts for the parquet, the modelling frame and the train / validation / test splits are
printed at run time; the development set is train + validation, all decisions are
cross-validated on it, and the test set is touched only by the Phase 6 evaluation cells.
Listing-date parts (`listing_year`, `listing_month`) are derived afterwards, and only join
the candidate pool when the dates span more than one year on the development set.


In [ ]:
# Cell 1.8 — Target filters (archived M1) + split (archived M3); NO null drop on features.
MIN_MODELLING_ROWS = 300   # hard floor: fewer modelling rows than this -> stop

# M1 — the archived target filters: keep rows that can be used for regression.
df_model = df_full[df_full["size_acres"] > 0].copy()
df_model = df_model[df_model["price_per_acre"] > 0].copy()
df_model = df_model[np.isfinite(df_model[TARGET])].copy()
df_model = df_model.reset_index(drop=True)
print(f"PHASE 1.8 — rows after the target filters (size_acres > 0, price_per_acre > 0, "
      f"finite target): {len(df_model):,} of {len(df_full):,}")

if len(df_model) < MIN_MODELLING_ROWS:
    raise RuntimeError(
        f"STOP: only {len(df_model)} modelling rows remain after the target filters "
        f"(minimum {MIN_MODELLING_ROWS}).\n"
        "Modelling on fewer rows is not meaningful. Check that DATA_PATH points at "
        "Notebook 1's engineered parquet and that the upstream filters behaved as expected."
    )

# The archived M2 null-drop on the 9 base features is deliberately NOT applied:
# dist_to_water_body_km is missing BY DESIGN for counties that do not border a water
# body. Missing values are handled per model instead (native NaN for XGBoost/CatBoost;
# training-median imputation + missing indicators for OLS/TabNet — see Cell 1.9).
_required_cols = BASE_FEATURES + [TARGET]
_extra_cols = ["size_acres", "price_per_acre", "list_date", "latitude", "longitude",
               "prop_type", "county", "reference_city",
               "geocode_source", "infra_source", "size_source"]
df_model = df_model[_required_cols + _extra_cols].reset_index(drop=True)

# Derived candidate feature: 1 when a water-body distance exists, 0 when it is missing
# by design. Available at prediction time; Phase 3 decides whether it earns a slot.
df_model["has_water_body_distance"] = df_model["dist_to_water_body_km"].notna().astype(int)

# M3 — the two archived train_test_split calls (identical code, random_state=42).
dev_df, test_df = train_test_split(df_model, test_size=0.15, random_state=42)
train_df, val_df = train_test_split(dev_df, test_size=0.176, random_state=42)

print(f"Modelling frame : {len(df_model):,} rows (target filters only — no null drop)")
print(f"Development set : {len(dev_df):,} rows (train {len(train_df):,} + "
      f"validation {len(val_df):,})")
print(f"Test set        : {len(test_df):,} rows — read only in the Phase 6 evaluation cells")

# Listing-date candidate features: considered only when the dates span more than one year.
# The span check uses the DEVELOPMENT set only — test rows must not drive decisions.
_dates = pd.to_datetime(dev_df["list_date"], errors="coerce").dropna()
LISTING_DATE_ACTIVE = bool(len(_dates) >= 2 and (_dates.max() - _dates.min()).days > 365)
print(f"list_date span on development set: "
      f"{(_dates.max() - _dates.min()).days if len(_dates) >= 2 else 0} days -> "
      f"listing_year/listing_month {'ADDED TO' if LISTING_DATE_ACTIVE else 'excluded from'} "
      "the Phase 3 candidate pool")
if LISTING_DATE_ACTIVE:
    for _frame in (df_model, dev_df, train_df, val_df, test_df):
        _dt = pd.to_datetime(_frame["list_date"], errors="coerce")
        _frame["listing_year"] = _dt.dt.year.astype(float)
        _frame["listing_month"] = _dt.dt.month.astype(float)
        # rows with an unparsable date keep NaN -> handled by fold-fitted imputation later

<a id="cell-1-9"></a>

### Cell 1.9 — Data-Quality Snapshot

The modelling frame keeps rows with missing features, so the missingness must be quantified
before any model sees it. This cell prints, per candidate feature, how many modelling rows are
null — `dist_to_water_body_km` dominates, missing by design for counties that do not border a
water body — a five-number target summary (min, 1st, 50th, 99th percentile, max) and how many
rows fall outside the 1st–99th percentile window. That same window drives the Phase 6
outlier-trimmed sensitivity check, where the development percentiles are applied to the test
rows; those counts are printed here too.


In [ ]:
# Cell 1.9 — Missingness per feature + target distribution on the modelling frame.
_dq_cols = (BASE_FEATURES + ["has_water_body_distance"]
            + ["latitude", "longitude", "prop_type", "county", "reference_city",
               "geocode_source", "infra_source", "size_source"])
if LISTING_DATE_ACTIVE:
    _dq_cols += ["listing_year", "listing_month"]
_dq = pd.DataFrame({
    "feature": _dq_cols,
    "missing_rows": [int(df_model[c].isna().sum()) for c in _dq_cols],
})
_dq["missing_%"] = (100.0 * _dq["missing_rows"] / len(df_model)).round(2)
print(f"PHASE 1.9 — data-quality snapshot (modelling frame, {len(df_model):,} rows)")
print(_dq.to_string(index=False))

_t = df_model[TARGET]
_t_min, _t_p1, _t_p50, _t_p99, _t_max = np.percentile(_t, [0, 1, 50, 99, 100])
_outside = int(((_t < _t_p1) | (_t > _t_p99)).sum())
print(f"\nTarget summary on the modelling frame (log price per acre):")
print(f"  min {_t_min:.4f} | p1 {_t_p1:.4f} | p50 {_t_p50:.4f} | "
      f"p99 {_t_p99:.4f} | max {_t_max:.4f}")
print(f"  rows outside the 1st-99th percentile window: {_outside} of {len(df_model):,}")

# The Phase 6 sensitivity check applies the DEVELOPMENT percentiles to the test rows;
# these counts say in advance how many that removes.
_dev_p1, _dev_p99 = np.percentile(dev_df[TARGET], [1, 99])
print(f"  development window [{_dev_p1:.4f}, {_dev_p99:.4f}]: "
      f"{int(((dev_df[TARGET] < _dev_p1) | (dev_df[TARGET] > _dev_p99)).sum())} of {len(dev_df):,} "
      f"dev rows outside; "
      f"{int(((test_df[TARGET] < _dev_p1) | (test_df[TARGET] > _dev_p99)).sum())} of "
      f"{len(test_df):,} test rows outside")

<a id="cell-1-10"></a>

### Cell 1.10 — Metrics Helper

`R2`, `RMSE` and `MAE` are computed on the log scale (the modelling scale). Because
`exp()` of a log-scale prediction returns the **median** of the log-normal price distribution
(not the mean), the two money-scale metrics are labelled accordingly: median absolute
percentage error and the mean absolute error of those median estimates in KSh per acre.


In [ ]:
# Cell 1.10 — Metric helpers (log scale + median-based money scale).
def rmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.mean(np.abs(y_true - y_pred)))

def r2_score_manual(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    return float(1.0 - ss_res / ss_tot) if ss_tot > 0 else float("nan")

def regression_metrics(y_true_log, y_pred_log):
    """R2/RMSE/MAE on the log scale; MedAPE and KSh-per-acre MAE on exp() of them.

    exp(log_pred) is the MEDIAN price per acre of the implied log-normal
    distribution, not its mean — stated wherever these money metrics appear.
    """
    y_true_log = np.asarray(y_true_log, float)
    y_pred_log = np.asarray(y_pred_log, float)
    ppa_true = np.exp(y_true_log)
    ppa_pred = np.exp(y_pred_log)  # median estimates
    return {
        "R2": r2_score_manual(y_true_log, y_pred_log),
        "RMSE": rmse(y_true_log, y_pred_log),
        "MAE": mae(y_true_log, y_pred_log),
        "MedAPE_%": float(np.median(np.abs(ppa_pred - ppa_true) / ppa_true) * 100),
        "MAE_KSh_per_acre": float(np.mean(np.abs(ppa_pred - ppa_true))),
    }

def format_metrics(m):
    return (f"R2 {m['R2']:.4f} | RMSE {m['RMSE']:.4f} | MAE {m['MAE']:.4f} | "
            f"MedAPE {m['MedAPE_%']:.1f}% | MAE {m['MAE_KSh_per_acre']:,.0f} KSh/acre (median basis)")

print("PHASE 1.10 — metrics helper ready.")
phase_timer_print("Phase 1 — setup and data")

<a id="phase-2"></a>

## Phase 2 — Baselines

Two references every later model must beat: an OLS on the 9 base features (scaler fitted on
the training split only, as in the archived Cells M4–M5) and a predict-the-training-median
baseline. Both are reported on the validation split here; the Phase 6 tables re-fit them on
the full development set so they compare like-for-like against the tree models.


<a id="cell-2-1"></a>

### Cell 2.1 — OLS Baseline

`fit_ols` is a reusable estimator: **training-median imputation with missing-indicator
columns** + `StandardScaler` for numeric columns and one-hot encoding for categoricals,
everything fitted on the training rows it is given and nothing else. The indicators let OLS
see where a value was imputed — mostly `dist_to_water_body_km`, missing by design for
counties that do not border a water body. Phase 6 re-uses this estimator per run.


In [ ]:
# Cell 2.1 — OLS baseline on the 9 base features; scaler fitted on the training split only.
phase_timer_start()   # Phase 2 starts here; Cell 2.2 prints its elapsed time.

def fit_ols(train_frame, features):
    """OLS pipeline fitted on `train_frame` only.

    Numeric columns: training-median imputation WITH missing-indicator columns
    (training statistics) + StandardScaler (training statistics). Categorical columns:
    one-hot with handle_unknown='ignore'. No statistic is ever computed on validation
    or test rows.
    """
    feats = list(features)
    num_cols = [c for c in feats if pd.api.types.is_numeric_dtype(train_frame[c])]
    cat_cols = [c for c in feats if c not in num_cols]
    steps = [("num", Pipeline([("impute", SimpleImputer(strategy="median",
                                                        add_indicator=True,
                                                        keep_empty_features=True)),
                               ("scale", StandardScaler())]), num_cols)]
    if cat_cols:
        steps.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols))
    pre = ColumnTransformer(steps)
    pipe = Pipeline([("pre", pre), ("ols", LinearRegression())])
    pipe.fit(train_frame[feats], train_frame[TARGET].values)
    return pipe

ols_baseline = fit_ols(train_df, BASE_FEATURES)
_pred_val = ols_baseline.predict(val_df[BASE_FEATURES])
print("PHASE 2.1 — OLS baseline (9 base features, scaler on train only)")
print(f"  Validation: {format_metrics(regression_metrics(val_df[TARGET].values, _pred_val))}")

# The same estimator re-fitted on all development rows — the row set the Phase 6 models
# train on — so the final comparison table is like-for-like.
ols_dev = fit_ols(dev_df, BASE_FEATURES)
_pred_val_dev = ols_dev.predict(val_df[BASE_FEATURES])
print(f"  Validation (re-fitted on dev): "
      f"{format_metrics(regression_metrics(val_df[TARGET].values, _pred_val_dev))}")

<a id="cell-2-2"></a>

### Cell 2.2 — Training-Median Baseline

Predicts the training split's median log price per acre for every row — the floor any learned
model must beat. `exp()` of it is the training median in KSh per acre.


In [ ]:
# Cell 2.2 — Predict-the-training-median baseline (log scale).
TRAIN_MEDIAN_LOG = float(train_df[TARGET].median())
TRAIN_MEDIAN_KSH = float(np.exp(TRAIN_MEDIAN_LOG))

_pred_med = np.full(len(val_df), TRAIN_MEDIAN_LOG)
print("PHASE 2.2 — training-median baseline")
print(f"  Training median (log)      : {TRAIN_MEDIAN_LOG:.4f}")
print(f"  Training median (KSh/acre) : {TRAIN_MEDIAN_KSH:,.0f}")
print(f"  Validation: {format_metrics(regression_metrics(val_df[TARGET].values, _pred_med))}")
phase_timer_print("Phase 2 — baselines")

<a id="phase-3"></a>

## Phase 3 — Feature Evaluation (development set only)

Evaluates the candidate feature pool **on the development set only** — the test set plays no
role here. Steps: define the pool and mark each feature as available at prediction time or not
(quality flags and provenance columns are "No": they describe how a row was produced, not the
listing itself; "No" features are still measured, but only "Yes" features may enter the final
set); pre-screen for missingness, variance, pairwise Spearman correlation above 0.9 and mutual
information; run a group ablation with XGBoost on repeated K-fold; permutation importance on
held-out folds and a SHAP screening; then a greedy forward selection with a one-standard-error
acceptance rule, a backward check, a CatBoost confirmation, and a stability check across CV
repeats. Cell 3.8 prints one ranked table and fills `SELECTED_FEATURES` (the one-SE set) and
`ALTERNATIVE_FEATURES` (the minimum-CV-error set) from it.

Cell 3.2 also defines the preprocessing used by every later phase (rare categories grouped
into "Other" with training data only, per-fold imputation medians, TabNet's numeric view) and
the single shared fold plan — identical folds for every model and both runs.


<a id="cell-3-1"></a>

### Cell 3.1 — Candidate Pool and Availability

The pool is the 9 base features plus the derived `has_water_body_distance` indicator (Cell 1.8;
1 = a water-body distance exists), coordinates, categorical location columns, provenance /
quality columns, and the listing-date parts when active. Price columns, the target, price
flags, identifiers and free-text columns never enter — they leak the answer or identify a row.


In [ ]:
# Cell 3.1 — Candidate pool + availability-at-prediction-time flags.
phase_timer_start()   # Phase 3 starts here; Cell 3.8 prints its elapsed time.
CANDIDATE_POOL = list(BASE_FEATURES) + [
    "has_water_body_distance",   # derived in Cell 1.8: 1 = dist_to_water_body_km present
    "latitude", "longitude",
    "prop_type", "county", "reference_city",
    "geocode_source", "infra_source", "size_source",
]
if LISTING_DATE_ACTIVE:
    CANDIDATE_POOL += ["listing_year", "listing_month"]

CATEGORICAL_CANDIDATES = ["prop_type", "county", "reference_city",
                          "geocode_source", "infra_source", "size_source"]

# Available at prediction time? "Yes" = a user (or the pipeline run on user-supplied inputs)
# can provide it for a NEW listing. Quality flags and data-source provenance describe HOW a
# row was produced — "No". They are still evaluated below to measure their effect, but only
# "Yes" features may enter the final selected set. has_water_body_distance is "Yes": a new
# listing either has a computed water-body distance or it does not.
AVAILABILITY = {c: "Yes" for c in CANDIDATE_POOL}
for _c in ["geocode_confidence", "geocode_source", "infra_source", "size_source"]:
    AVAILABILITY[_c] = "No"

# Columns that must NEVER be features: price family, the target, identifiers, free text.
NEVER_FEATURES = ["price_ksh", "log_price_ksh", "price_per_acre", "price_flag", TARGET,
                  "listing_url", "listing_no", "title", "location"]

_missing = [c for c in CANDIDATE_POOL if c not in dev_df.columns]
if _missing:
    raise ValueError(f"Candidate features missing from the data: {_missing}")
_overlap = [c for c in CANDIDATE_POOL if c in NEVER_FEATURES]
assert not _overlap, f"Candidate pool overlaps the never-use list: {_overlap}"

ELIGIBLE_POOL = [c for c in CANDIDATE_POOL if AVAILABILITY[c] == "Yes"]

print("PHASE 3.1 — candidate pool (development set only)")
print(pd.DataFrame({
    "feature": CANDIDATE_POOL,
    "role": ["categorical" if c in CATEGORICAL_CANDIDATES else "numeric" for c in CANDIDATE_POOL],
    "available_at_prediction_time": [AVAILABILITY[c] for c in CANDIDATE_POOL],
}).to_string(index=False))
print(f"\nEligible (Yes) features that may enter the final set ({len(ELIGIBLE_POOL)}): "
      f"{ELIGIBLE_POOL}")
print(f"Measured but not eligible (No): "
      f"{[c for c in CANDIDATE_POOL if AVAILABILITY[c] == 'No']}")

<a id="cell-3-2"></a>

### Cell 3.2 — Preprocessing and Fold Plan

`PipelinePreprocessor` is fitted on **training rows only** — per fold inside cross-validation,
on the development set for the final refits. It groups categories with fewer than
`MIN_CATEGORY_ROWS` training rows into `"Other"`, stores the level lists, and produces the
three views the models need: XGBoost and CatBoost keep **numeric NaNs untouched** (both
learners handle missing values natively and learn a split direction for them), while
TabNet's numeric view imputes with training medians, adds missing-value indicator columns and
standardises with training statistics (raw-scale inputs make TabNet's slowly-adapting BatchNorm
running statistics diverge on small data). The shared fold plan built here is used unchanged by
every model, every phase and both runs.


In [ ]:
# Cell 3.2 — Shared preprocessing + the single fold plan for the whole notebook.
MIN_CATEGORY_ROWS = 5   # categories with fewer training rows than this become "Other"

class PipelinePreprocessor:
    """Fit-on-training-rows preprocessing producing model-specific feature matrices.

    fit() learns, from the training frame only:
      - numeric medians (for the TabNet imputation) and which numeric columns need a
        missing-value indicator column for TabNet,
      - category level lists after grouping rare levels (< MIN_CATEGORY_ROWS) into
        "Other" (unseen values at transform time also map to "Other"),
      - label encodings for the TabNet view.
    transform_xgb()      -> DataFrame, categoricals as pandas category dtype;
                            numeric NaNs KEPT (XGBoost handles them natively)
    transform_catboost() -> DataFrame, categoricals as plain strings;
                            numeric NaNs KEPT (CatBoost handles them natively)
    transform_tabnet()   -> all-numeric DataFrame: label-encoded categoricals,
                            numerics imputed with training medians and standardised
                            with training statistics, plus missing-indicator columns
    """

    def __init__(self, feature_cols, categorical_cols, min_category_rows=MIN_CATEGORY_ROWS):
        self.feature_cols = list(feature_cols)
        self.categorical_cols = [c for c in categorical_cols if c in self.feature_cols]
        self.numeric_cols = [c for c in self.feature_cols if c not in self.categorical_cols]
        self.min_category_rows = int(min_category_rows)

    def fit(self, frame):
        d = frame[self.feature_cols].copy()
        self.medians_ = {}
        self.missing_indicator_cols_ = []
        for c in self.numeric_cols:
            self.medians_[c] = float(d[c].median()) if d[c].notna().any() else 0.0
            if d[c].isna().any():
                self.missing_indicator_cols_.append(c + "_was_missing")
        # TabNet numeric input standardisation (training statistics only). Raw-scale inputs
        # with TabNet's slowly-adapting BatchNorm running statistics diverge on small folds;
        # standardised inputs keep the network in a sane range whatever the batch size.
        self.tabnet_stats_ = {}
        for c in self.numeric_cols:
            filled = d[c].fillna(self.medians_[c])
            s = float(filled.std(ddof=0)) if len(filled) > 1 else 0.0
            self.tabnet_stats_[c] = [float(filled.mean()), s if s > 1e-8 else 1.0]
        self.levels_ = {}
        self.group_maps_ = {}
        self.label_maps_ = {}
        for c in self.categorical_cols:
            vals = d[c].fillna("Missing").astype(str)
            vc = vals.value_counts()
            keep = sorted(vc[vc >= self.min_category_rows].index.tolist())
            self.levels_[c] = keep + ["Other"]
            self.group_maps_[c] = {v: (v if v in keep else "Other") for v in vc.index}
            self.label_maps_[c] = {v: i for i, v in enumerate(self.levels_[c])}
        return self

    def _clean(self, frame, impute_numeric=False):
        """Categoricals -> grouped strings; numeric NaNs kept unless impute_numeric."""
        out = frame.reindex(columns=self.feature_cols).copy()
        for c in self.categorical_cols:
            out[c] = (out[c].fillna("Missing").astype(str)
                      .map(lambda v, c=c: self.group_maps_[c].get(v, "Other")))
        if impute_numeric:
            for c in self.numeric_cols:
                out[c] = out[c].fillna(self.medians_[c])
        return out

    def transform_xgb(self, frame):
        out = self._clean(frame)          # numeric NaNs kept -> native handling
        for c in self.categorical_cols:
            out[c] = pd.Categorical(out[c], categories=self.levels_[c])
        return out

    def transform_catboost(self, frame):
        return self._clean(frame)         # numeric NaNs kept -> native handling

    def transform_tabnet(self, frame):
        out = self._clean(frame, impute_numeric=True)
        raw = frame.reindex(columns=self.feature_cols).copy()
        for c in self.categorical_cols:
            out[c] = out[c].map(self.label_maps_[c]).astype(float)
        for c in self.missing_indicator_cols_:
            out[c] = raw[c[:-len("_was_missing")]].isna().astype(float)
        for c in self.numeric_cols:                    # training-statistic standardisation
            m, s = self.tabnet_stats_[c]
            out[c] = (out[c] - m) / s
        return out

    def tabnet_cat_info(self):
        """(cat_idxs, cat_dims) for TabNet, in transform_tabnet() column order."""
        cols = self.feature_cols + self.missing_indicator_cols_
        idxs = [cols.index(c) for c in self.categorical_cols]
        dims = [len(self.levels_[c]) for c in self.categorical_cols]
        return idxs, dims

    def to_dict(self):
        return {
            "feature_cols": self.feature_cols,
            "categorical_cols": self.categorical_cols,
            "numeric_cols": self.numeric_cols,
            "min_category_rows": self.min_category_rows,
            "medians": self.medians_,
            "missing_indicator_cols": self.missing_indicator_cols_,
            "tabnet_stats": self.tabnet_stats_,
            "levels": self.levels_,
            "group_maps": self.group_maps_,
            "label_maps": self.label_maps_,
        }

    @classmethod
    def from_dict(cls, d):
        obj = cls(d["feature_cols"], d["categorical_cols"], d["min_category_rows"])
        obj.medians_ = {k: float(v) for k, v in d["medians"].items()}
        obj.missing_indicator_cols_ = list(d["missing_indicator_cols"])
        obj.tabnet_stats_ = {k: [float(v[0]), float(v[1])] for k, v in d["tabnet_stats"].items()}
        obj.levels_ = d["levels"]
        obj.group_maps_ = d["group_maps"]
        obj.label_maps_ = d["label_maps"]
        return obj

# The single shared fold plan: identical folds for every model, every phase, both runs.
SHARED_FOLDS = list(RepeatedKFold(n_splits=N_FOLDS, n_repeats=CV_REPEATS,
                                  random_state=SEED).split(dev_df))
print(f"PHASE 3.2 — preprocessing ready; shared fold plan: "
      f"{CV_REPEATS} repeats x {N_FOLDS} folds = {len(SHARED_FOLDS)} folds on "
      f"{len(dev_df)} development rows")

# Fixed-budget XGBoost used ONLY to compare feature configurations (not a final model).
SCREEN_XGB_PARAMS = dict(
    n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.9,
    colsample_bytree=0.9, reg_lambda=1.0, tree_method="hist",
    enable_categorical=True, random_state=SEED, early_stopping_rounds=50, verbosity=0,
)

def make_screen_xgb(with_early_stopping=True):
    params = dict(SCREEN_XGB_PARAMS)
    if not with_early_stopping:
        params.pop("early_stopping_rounds")
    return xgb.XGBRegressor(**params)

def xgb_cv(dev_frame, features, folds, with_early_stopping=True):
    """Repeated-K-fold RMSE (mean, se) of the screening XGB for one feature set."""
    cat_cols = [c for c in features if c in CATEGORICAL_CANDIDATES]
    y = dev_frame[TARGET].values
    scores = []
    for tr, va in folds:
        prep = PipelinePreprocessor(features, cat_cols).fit(dev_frame.iloc[tr])
        model = make_screen_xgb(with_early_stopping)
        model.fit(prep.transform_xgb(dev_frame.iloc[tr]), y[tr],
                  eval_set=[(prep.transform_xgb(dev_frame.iloc[va]), y[va])], verbose=False)
        scores.append(rmse(y[va], model.predict(prep.transform_xgb(dev_frame.iloc[va]))))
    return float(np.mean(scores)), float(np.std(scores, ddof=1) / np.sqrt(len(scores)))

print("Screening XGB and CV helper ready.")

<a id="cell-3-3"></a>

### Cell 3.3 — Pre-Screen

Missing percentage, variance / category concentration, Spearman pairs above 0.9, and mutual
information against the target — all on the development set. Categorical mutual information is
computed on factorised codes and is a rough ranking aid only.


In [ ]:
# Cell 3.3 — Pre-screen: missing %, variance, Spearman pairs > 0.9, mutual information.
_rows = []
for c in CANDIDATE_POOL:
    s = dev_df[c]
    row = {"feature": c, "available": AVAILABILITY[c],
           "missing_%": round(100.0 * float(s.isna().mean()), 2)}
    if c in CATEGORICAL_CANDIDATES:
        vc = s.fillna("Missing").astype(str).value_counts()
        row.update(n_unique=int(vc.size), variance=np.nan,
                   top_level_share=round(float(vc.iloc[0] / vc.sum()), 3))
    else:
        row.update(n_unique=int(s.nunique()), variance=round(float(s.var()), 4),
                   top_level_share=np.nan)
    _rows.append(row)
prescreen = pd.DataFrame(_rows)
print("PHASE 3.3 — pre-screen (development set)")
print(prescreen.to_string(index=False))

_num_pool = [c for c in CANDIDATE_POOL if c not in CATEGORICAL_CANDIDATES]
_corr = dev_df[_num_pool].corr(method="spearman")
print("\nSpearman pairs with |rho| > 0.9:")
_found = False
for i, a in enumerate(_num_pool):
    for b in _num_pool[i + 1:]:
        r = float(_corr.loc[a, b])
        if abs(r) > 0.9:
            print(f"  {a} ~ {b}: rho = {r:.3f}")
            _found = True
if not _found:
    print("  none")

_mi_rows = []
_X_mi = dev_df[_num_pool].fillna(dev_df[_num_pool].median())
_mi_num = mutual_info_regression(_X_mi, dev_df[TARGET].values, random_state=SEED)
for c, v in zip(_num_pool, _mi_num):
    _mi_rows.append((c, float(v)))
for c in CATEGORICAL_CANDIDATES:
    codes = dev_df[c].fillna("Missing").astype(str).astype("category").cat.codes
    v = float(mutual_info_regression(codes.to_frame(), dev_df[TARGET].values,
                                     random_state=SEED)[0])
    _mi_rows.append((c, v))
mi_table = (pd.DataFrame(_mi_rows, columns=["feature", "mutual_information"])
            .sort_values("mutual_information", ascending=False)
            .reset_index(drop=True))
print("\nMutual information vs target (ranked; categoricals on factorised codes — "
      "ranking aid only):")
print(mi_table.to_string(index=False))

<a id="cell-3-4"></a>

### Cell 3.4 — Group Ablation

XGBoost with repeated K-fold on the shared folds, for each feature **group alone** and
leave-one-group-out, plus the full pool. Groups: size, coordinates, categorical location,
proximity, pillar scores, quality flags (and listing date when active). This shows which
information blocks carry signal without depending on any single feature.


In [ ]:
# Cell 3.4 — Group ablation with XGBoost (repeated K-fold, shared folds).
FEATURE_GROUPS = {
    "size":                 ["log_size_acres"],
    "coordinates":          ["latitude", "longitude"],
    "categorical_location": ["prop_type", "county", "reference_city"],
    "proximity":            ["dist_to_nairobi_km", "dist_to_county_town_km",
                             "dist_to_water_body_km", "has_water_body_distance",
                             "reference_city_dist_km"],
    "pillar_scores":        ["amenities_score", "accessibility_score", "infrastructure_score"],
    "quality_flags":        ["geocode_confidence", "geocode_source", "infra_source", "size_source"],
}
if LISTING_DATE_ACTIVE:
    FEATURE_GROUPS["listing_date"] = ["listing_year", "listing_month"]

_configs = {"all_pool": CANDIDATE_POOL}
for _g, _cols in FEATURE_GROUPS.items():
    _configs[f"only_{_g}"] = _cols
    _configs[f"drop_{_g}"] = [c for c in CANDIDATE_POOL if c not in _cols]

_ablation_rows = []
for _name, _feats in _configs.items():
    _m, _se = xgb_cv(dev_df, _feats, SHARED_FOLDS)
    _ablation_rows.append({"config": _name, "n_features": len(_feats),
                           "cv_rmse": round(_m, 4), "se": round(_se, 4)})
ablation_table = (pd.DataFrame(_ablation_rows)
                  .sort_values("cv_rmse").reset_index(drop=True))
print("PHASE 3.4 — group ablation (XGBoost, repeated K-fold on the development set)")
print(ablation_table.to_string(index=False))

<a id="cell-3-5"></a>

### Cell 3.5 — Permutation Importance

Held-out-fold permutation importance: within each shared fold, fit the screening XGB on the
training part, then measure how much shuffling each feature (several times, deterministic RNG)
worsens RMSE on the held-out part. Reported as the mean RMSE increase per feature.


In [ ]:
# Cell 3.5 — Permutation importance on held-out folds (manual, deterministic RNG).
PERM_N_REPEATS = 3 if FAST_DEBUG else 5

def permutation_importance_cv(dev_frame, features, folds, n_repeat=PERM_N_REPEATS, seed=SEED):
    rng = np.random.default_rng(seed)
    y = dev_frame[TARGET].values
    cat_cols = [c for c in features if c in CATEGORICAL_CANDIDATES]
    records = {c: [] for c in features}
    for k, (tr, va) in enumerate(folds):
        prep = PipelinePreprocessor(features, cat_cols).fit(dev_frame.iloc[tr])
        Xva = prep.transform_xgb(dev_frame.iloc[va])
        model = make_screen_xgb()
        model.fit(prep.transform_xgb(dev_frame.iloc[tr]), y[tr],
                  eval_set=[(Xva, y[va])], verbose=False)
        base = rmse(y[va], model.predict(Xva))
        for j, c in enumerate(features):
            deltas = []
            for _ in range(n_repeat):
                Xp = Xva.copy()
                shuffled = (Xva.iloc[:, j]
                            .sample(frac=1.0, random_state=int(rng.integers(2 ** 31)))
                            .reset_index(drop=True))
                Xp.isetitem(j, shuffled)   # Series keeps the categorical dtype intact
                deltas.append(rmse(y[va], model.predict(Xp)) - base)
            records[c].append(float(np.mean(deltas)))
    return {c: (float(np.mean(v)), float(np.std(v, ddof=1))) for c, v in records.items()}

perm_importance = permutation_importance_cv(dev_df, CANDIDATE_POOL, SHARED_FOLDS)
perm_table = (pd.DataFrame([{"feature": c, "mean_delta_rmse": round(m, 4),
                             "std_delta_rmse": round(s, 4)}
                            for c, (m, s) in perm_importance.items()])
              .sort_values("mean_delta_rmse", ascending=False).reset_index(drop=True))
print("PHASE 3.5 — permutation importance on held-out folds (mean RMSE increase when shuffled)")
print(perm_table.to_string(index=False))

<a id="cell-3-6"></a>

### Cell 3.6 — SHAP Screening

TreeSHAP on a screening XGBoost fitted on the whole development set (all pool features).
Saved to `outputs/modelling/feature_screening_shap.png`; the mean-absolute-SHAP ranking feeds
the Cell 3.8 table.


In [ ]:
# Cell 3.6 — SHAP screening (TreeExplainer on a development-set XGBoost).
prep_shap = PipelinePreprocessor(CANDIDATE_POOL, CATEGORICAL_CANDIDATES).fit(dev_df)
X_dev_shap = prep_shap.transform_xgb(dev_df)
shap_screen_model = make_screen_xgb(with_early_stopping=False)
shap_screen_model.fit(X_dev_shap, dev_df[TARGET].values, verbose=False)

explainer = shap.TreeExplainer(shap_screen_model)
shap_values = explainer.shap_values(X_dev_shap)

shap_mean_abs = {c: float(np.abs(v).mean())
                 for c, v in zip(X_dev_shap.columns, np.asarray(shap_values).T)}

plt.figure()
shap.summary_plot(np.asarray(shap_values), X_dev_shap, show=False, max_display=20)
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/feature_screening_shap.png", dpi=150, bbox_inches="tight")
plt.show()
print("PHASE 3.6 — SHAP screening (mean |SHAP|, ranked); chart saved to "
      f"{OUTPUTS}/feature_screening_shap.png")
for c, v in sorted(shap_mean_abs.items(), key=lambda kv: -kv[1]):
    print(f"  {c:28s} {v:.4f}")

<a id="cell-3-7"></a>

### Cell 3.7 — Greedy Selection and Stability

Greedy forward selection from the **eligible** pool with the screening XGBoost on the shared
folds: seed with the best single eligible feature, then repeatedly add the feature that lowers
CV RMSE the most, accepting it only when the improvement exceeds **one standard error** of the
candidate configuration's repeated-CV RMSE. A backward pass then drops any feature whose
removal does not worsen RMSE by more than one standard error. A CatBoost run confirms the
result against the 9 base features. Because repeated-CV feature-selection estimates at this
sample size are noisy, the whole search is repeated once per CV repeat, and only features
picked in the majority of passes (plus the main all-folds pass) survive.


In [ ]:
# Cell 3.7 — Greedy forward + backward check + CatBoost confirmation + stability.
_CV_CACHE = {}

def cached_cv(features, folds, fold_key):
    key = (fold_key, tuple(sorted(features)))
    if key not in _CV_CACHE:
        _CV_CACHE[key] = xgb_cv(dev_df, features, folds)
    return _CV_CACHE[key]

def greedy_forward(folds, fold_key):
    """Forward selection with the one-standard-error acceptance rule."""
    solo = {f: cached_cv([f], folds, fold_key)[0] for f in ELIGIBLE_POOL}
    seed = min(solo, key=solo.get)
    selected = [seed]
    current_m, _ = cached_cv([seed], folds, fold_key)
    improved = True
    while improved:
        improved = False
        best = None
        for f in [c for c in ELIGIBLE_POOL if c not in selected]:
            m, se = cached_cv(selected + [f], folds, fold_key)
            if best is None or m < best[1]:
                best = (f, m, se)
        if best is not None and best[1] < current_m - best[2]:   # improve by > 1 SE
            selected.append(best[0])
            current_m = best[1]
            improved = True
    return selected

def backward_check(selected, folds, fold_key):
    """Drop features whose removal does not worsen RMSE by more than one SE."""
    selected = list(selected)
    changed = True
    while changed:
        changed = False
        current_m, _ = cached_cv(selected, folds, fold_key)
        for f in list(selected):
            if len(selected) == 1:
                break
            m_rem, se_rem = cached_cv([c for c in selected if c != f], folds, fold_key)
            if m_rem <= current_m + se_rem:                     # not worth keeping
                selected = [c for c in selected if c != f]
                changed = True
                break
    return selected

# Main pass on all folds, then one pass per CV repeat for stability.
selection_main = backward_check(greedy_forward(SHARED_FOLDS, "all"), SHARED_FOLDS, "all")
selection_by_repeat = {}
for r in range(CV_REPEATS):
    folds_r = SHARED_FOLDS[r * N_FOLDS:(r + 1) * N_FOLDS]
    key = f"repeat{r}"
    selection_by_repeat[r] = backward_check(greedy_forward(folds_r, key), folds_r, key)

_votes = {f: 1 for f in selection_main}
for r, sel in selection_by_repeat.items():
    for f in sel:
        _votes[f] = _votes.get(f, 0) + 1
_majority = (CV_REPEATS + 1) / 2.0                       # main pass + repeats
STABLE_SELECTED = [f for f in selection_main if _votes.get(f, 0) >= _majority]
if not STABLE_SELECTED:
    STABLE_SELECTED = list(selection_main)               # never return an empty set

print("PHASE 3.7 — greedy selection (screening XGB, one-SE rule)")
print(f"  Main pass (all {len(SHARED_FOLDS)} folds): {selection_main}")
for r, sel in selection_by_repeat.items():
    print(f"  Repeat {r} (folds {r * N_FOLDS}-{(r + 1) * N_FOLDS - 1})      : {sel}")
print(f"  Votes (majority threshold >= {_majority:.1f} of {CV_REPEATS + 1} passes): {_votes}")
print(f"  STABLE SELECTED: {STABLE_SELECTED}")

# CatBoost confirmation on the shared folds: stable set vs the 9 base features.
def screen_catboost_cv(features):
    cat_cols = [c for c in features if c in CATEGORICAL_CANDIDATES]
    y = dev_df[TARGET].values
    scores = []
    for tr, va in SHARED_FOLDS:
        prep = PipelinePreprocessor(features, cat_cols).fit(dev_df.iloc[tr])
        model = CatBoostRegressor(iterations=600, learning_rate=0.05, depth=6,
                                  l2_leaf_reg=3.0, random_seed=SEED, verbose=0,
                                  early_stopping_rounds=100, allow_writing_files=False,
                                  cat_features=cat_cols)
        model.fit(prep.transform_catboost(dev_df.iloc[tr]), y[tr],
                  eval_set=(prep.transform_catboost(dev_df.iloc[va]), y[va]))
        scores.append(rmse(y[va], model.predict(prep.transform_catboost(dev_df.iloc[va]))))
    return float(np.mean(scores)), float(np.std(scores, ddof=1) / np.sqrt(len(scores)))

_cb_sel = screen_catboost_cv(STABLE_SELECTED)
_cb_base = screen_catboost_cv(BASE_FEATURES)
print(f"  CatBoost confirmation — stable set CV RMSE {_cb_sel[0]:.4f} (se {_cb_sel[1]:.4f}) "
      f"vs 9 base features {_cb_base[0]:.4f} (se {_cb_base[1]:.4f})")

<a id="cell-3-8"></a>

### Cell 3.8 — Ranked Table, SELECTED_FEATURES and ALTERNATIVE_FEATURES

One ranked table combining every Phase 3 view, `SELECTED_FEATURES` filled from it (the stable
one-SE set) and `ALTERNATIVE_FEATURES` — the minimum-CV-error set among all the feature
configurations the greedy search evaluated on the shared folds (eligible features only). The
alternative set is run through the full pipeline as **Run C** only when
`CONFIG.RUN_ALTERNATIVE` is True. If the stable set equals the 9 base features, Run B will
simply mirror Run A — the notebook notes this when it happens.


In [ ]:
# Cell 3.8 — Ranked table + SELECTED_FEATURES + ALTERNATIVE_FEATURES.
def _group_of(f):
    for g, cols in FEATURE_GROUPS.items():
        if f in cols:
            return g
    return "other"

ranked_rows = []
for f in CANDIDATE_POOL:
    ranked_rows.append({
        "feature": f,
        "group": _group_of(f),
        "available": AVAILABILITY[f],
        "missing_%": round(100.0 * float(dev_df[f].isna().mean()), 2),
        "mutual_info": round(float(mi_table.set_index("feature").loc[f, "mutual_information"]), 4)
                       if f in set(mi_table["feature"]) else np.nan,
        "perm_delta_rmse": round(perm_importance[f][0], 4),
        "mean_abs_shap": round(shap_mean_abs.get(f, np.nan), 4),
        "selection_votes": _votes.get(f, 0),
        "in_selected_set": f in STABLE_SELECTED,
    })
ranked_table = (pd.DataFrame(ranked_rows)
                .sort_values(["in_selected_set", "selection_votes", "perm_delta_rmse"],
                             ascending=[False, False, False])
                .reset_index(drop=True))
print("PHASE 3.8 — ranked feature table (development set only)")
print(ranked_table.to_string(index=False))

SELECTED_FEATURES = list(STABLE_SELECTED)     # <- Run B feature set (one-SE rule)
print(f"\nSELECTED_FEATURES (Run B, one-SE rule) = {SELECTED_FEATURES}")
if sorted(SELECTED_FEATURES) == sorted(BASE_FEATURES):
    print("Note: the selected set equals the 9 base features — Run B will mirror Run A.")
print("This list overrides the CONFIG default; hard-code CONFIG.SELECTED_FEATURES to skip "
      "the Phase 3 search on re-runs.")

# ALTERNATIVE_FEATURES: the minimum-CV-error set among all evaluated configurations
# (main pass, shared folds, eligible pool only). Run as Run C only when
# CONFIG.RUN_ALTERNATIVE is True (Cell 6.5).
_alt_evals = [(v[0], list(k[1])) for k, v in _CV_CACHE.items() if k[0] == "all"]
_alt_rmse, ALTERNATIVE_FEATURES = min(_alt_evals, key=lambda kv: kv[0])
_sel_rmse = cached_cv(SELECTED_FEATURES, SHARED_FOLDS, "all")[0]
print(f"\nALTERNATIVE_FEATURES (minimum CV error) = {ALTERNATIVE_FEATURES}")
print(f"  CV RMSE {_alt_rmse:.4f} vs the one-SE selected set {_sel_rmse:.4f}")
if sorted(ALTERNATIVE_FEATURES) == sorted(SELECTED_FEATURES):
    print("  Note: the alternative set equals the selected set — Run C would mirror Run B.")
elif sorted(ALTERNATIVE_FEATURES) == sorted(BASE_FEATURES):
    print("  Note: the alternative set equals the 9 base features — Run C would mirror Run A.")
phase_timer_print("Phase 3 — feature evaluation")

<a id="phase-4"></a>

## Phase 4 — Base Models

Optuna tuning of the three learners on the shared repeated-K-fold plan (identical folds for
all models and both runs, identical trial budgets):

- **XGBoost** — `hist` tree method, native categorical support, native NaN handling
  (numeric missingness passes straight through), early stopping, strong regularisation
  (`reg_alpha`, `reg_lambda`, `gamma`, `min_child_weight`, subsampling), and a tuned loss:
  squared error vs pseudo-Huber.
- **CatBoost** — `cat_features`, native NaN handling for numeric features, early stopping,
  and a tuned loss: RMSE vs Huber (delta tuned).
- **TabNet** — target standardised with training-fold mean/std and predictions inverted back;
  `y` shaped `(-1, 1)`; categorical columns label-encoded with embeddings (Cell 3.2 view);
  numeric NaNs filled with training medians plus missing-indicator columns, inputs
  standardised with training statistics; early stopping; GPU used automatically when
  available.

Every `tune_*` study is stored in a SQLite file under `models/runs/<run>/` and opened with
`load_if_exists=True`: a study that already holds the full trial budget is skipped, a partial
one is topped up, so interrupted runs resume. Rare categories are grouped into "Other" with
training data only (Cell 3.2). Each cell also defines an `*_oof` function (out-of-fold
predictions and the per-fold iteration counts with the tuned parameters); Phase 5 saves the
refit stage per run the same way. Phases 4–6 are packaged into one reusable routine (Cell 6.3)
that is called once per run.


<a id="cell-4-1"></a>

### Cell 4.1 — County Group Folds

A county-grouped `GroupKFold` plan for the geographic-honesty check reported next to the
standard CV score in Phase 6: whole counties move together between training and validation,
so the score measures generalisation to unseen counties.


In [ ]:
# Cell 4.1 — County-grouped folds (reported in Phase 6 next to the standard CV score).
phase_timer_start()   # Phase 4 starts here; Cell 4.4 prints its elapsed time.
N_GROUP_SPLITS = int(min(N_FOLDS, dev_df["county"].nunique()))
GROUP_FOLDS = list(GroupKFold(n_splits=N_GROUP_SPLITS)
                   .split(dev_df[["county"]], groups=dev_df["county"].values))
print(f"PHASE 4.1 — GroupKFold by county: {N_GROUP_SPLITS} splits over "
      f"{dev_df['county'].nunique()} counties, {len(dev_df)} development rows")
for i, (tr, va) in enumerate(GROUP_FOLDS):
    print(f"  split {i}: train {len(tr)} rows / held-out {len(va)} rows")

<a id="cell-4-2"></a>

### Cell 4.2 — XGBoost Tuning

Optuna over the shared folds; every fold fits its own preprocessor (training rows only) and
early stopping watches the fold's held-out part. The study lives in a SQLite file
`models/runs/<run>/optuna_xgb_<signature>.db` (opened with `load_if_exists=True`) and is
skipped when it already holds `N_TRIALS_XGB` finished trials. `study.best_params` is returned
per run; `xgb_oof` re-runs the folds with the tuned parameters to collect out-of-fold
predictions (averaged over repeats), per-fold best-iteration counts and the CV score — the
result is saved to `oof_xgb.json` and skipped when that file already matches the current
configuration.


In [ ]:
# Cell 4.2 — XGBoost: resumable Optuna tuning + tuned-parameter OOF collection.
XGB_MAX_ROUNDS = 4000

# --- resumable-stage helpers (used by every tune_* / *_oof / refit stage) ---------------
def _stage_signature(payload):
    """Short deterministic signature of a stage's configuration (sorted-key JSON -> sha1).

    A saved stage file is only reused when its stored signature matches the signature
    of the CURRENT configuration (features, tuned params, fold plan, seed), so stale
    artefacts are never silently reused after a change.
    """
    return hashlib.sha1(json.dumps(payload, sort_keys=True, default=str)
                        .encode("utf-8")).hexdigest()[:12]

def _to_jsonable(obj):
    """numpy -> plain JSON (arrays tagged, so they can be restored exactly)."""
    if isinstance(obj, np.ndarray):
        return {"__ndarray__": obj.tolist()}
    if isinstance(obj, dict):
        return {k: _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    return obj

def _from_jsonable(obj):
    if isinstance(obj, dict) and "__ndarray__" in obj:
        return np.asarray(obj["__ndarray__"], float)
    if isinstance(obj, dict):
        return {k: _from_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_from_jsonable(v) for v in obj]
    return obj

def _tune_with_resume(model_key, run_dir, objective, n_trials, label, features,
                      force_rerun=False):
    """Optuna study in a SQLite file (on Drive), load_if_exists=True.

    The study file name carries a signature of (model, features, fold plan, seed), so a
    study is only reused for the exact configuration that produced it. A study that
    already holds >= n_trials COMPLETE trials is skipped; a partial one is topped up.
    """
    sig = _stage_signature({"kind": "tune", "model": model_key,
                            "features": sorted(features), "n_folds": N_FOLDS,
                            "repeats": CV_REPEATS, "seed": SEED})
    db_path = os.path.join(run_dir, f"optuna_{model_key}_{sig}.db")
    if force_rerun and os.path.exists(db_path):
        os.remove(db_path)
        print(f"  [{label}] {model_key}: FORCE_RERUN — deleted {os.path.basename(db_path)}")
    study = optuna.create_study(
        study_name=f"{model_key}_{sig}", direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        storage=f"sqlite:///{db_path}", load_if_exists=True)
    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.get_trials())
    todo = n_trials - done
    if todo <= 0:
        print(f"  [{label}] {model_key}: study already holds {done} trials "
              f"(>= {n_trials}) — tuning SKIPPED; best CV RMSE {study.best_value:.4f}")
    else:
        if done:
            print(f"  [{label}] {model_key}: resuming study — {done} trials on file, "
                  f"adding {todo}")
        t_tune = time.time()
        study.optimize(objective, n_trials=todo, show_progress_bar=False)
        print(f"  [{label}] {model_key}: best CV RMSE {study.best_value:.4f} "
              f"({time.time() - t_tune:.1f}s)")
    print(f"    best params: {study.best_params}")
    return study.best_params

def _oof_with_resume(model_key, oof_fn, features, best_params, run_name, run_dir,
                     force_rerun=False):
    """Run (or load) one model's out-of-fold stage; results saved to oof_<model>.json."""
    sig_payload = {"kind": "oof", "model": model_key, "features": sorted(features),
                   "best_params": best_params, "n_folds": N_FOLDS,
                   "repeats": CV_REPEATS, "seed": SEED}
    sig = _stage_signature(sig_payload)
    path = os.path.join(run_dir, f"oof_{model_key}.json")
    if not force_rerun and os.path.exists(path):
        with open(path) as f:
            saved = json.load(f)
        if saved.get("signature") == sig:
            res = _from_jsonable(saved["result"])
            res["signature"] = sig
            print(f"  [{run_name}] {model_key}: OOF stage found on file "
                  f"({os.path.basename(path)}) — SKIPPED (CV RMSE {res['cv_rmse']:.4f})")
            return res
    t_oof = time.time()
    res = oof_fn(features, best_params, run_name)
    res["signature"] = sig
    with open(path, "w") as f:
        json.dump({"signature": sig,
                   "result": _to_jsonable({k: v for k, v in res.items()
                                           if k != "signature"})}, f)
    print(f"  [{run_name}] {model_key}: OOF stage computed and saved to "
          f"{os.path.basename(path)} ({time.time() - t_oof:.1f}s)")
    return res
# ----------------------------------------------------------------------------------------

def _xgb_params_from(best_params):
    """Convert study params to XGBRegressor kwargs (adds the fixed settings)."""
    p = dict(best_params)
    loss = p.pop("loss")
    params = {
        "objective": f"reg:{loss}",
        "n_estimators": XGB_MAX_ROUNDS,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": SEED,
        "verbosity": 0,
    }
    params.update(p)          # learning_rate, depth, regularisation, (huber_slope)
    return params

def tune_xgb(features, n_trials, run_name, run_dir, force_rerun=False):
    """Optuna tuning with a resumable SQLite study; returns best_params."""
    cat_cols = [c for c in features if c in CATEGORICAL_CANDIDATES]
    y = dev_df[TARGET].values

    def objective(trial):
        loss = trial.suggest_categorical("loss", ["squarederror", "pseudohubererror"])
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 50.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "early_stopping_rounds": 60,
        }
        if loss == "pseudohubererror":
            params["huber_slope"] = trial.suggest_float("huber_slope", 0.2, 2.0)
        model = xgb.XGBRegressor(**_xgb_params_from({"loss": loss, **params}))
        scores = []
        for tr, va in SHARED_FOLDS:
            prep = PipelinePreprocessor(features, cat_cols).fit(dev_df.iloc[tr])
            model.fit(prep.transform_xgb(dev_df.iloc[tr]), y[tr],
                      eval_set=[(prep.transform_xgb(dev_df.iloc[va]), y[va])], verbose=False)
            scores.append(rmse(y[va], model.predict(prep.transform_xgb(dev_df.iloc[va]))))
        trial.set_user_attr("fold_rmse", scores)
        return float(np.mean(scores))

    return _tune_with_resume("xgb", run_dir, objective, n_trials, run_name, features,
                             force_rerun=force_rerun)

def xgb_oof(features, best_params, run_name):
    """Tuned-parameter OOF predictions (mean over repeats) + per-fold best rounds."""
    cat_cols = [c for c in features if c in CATEGORICAL_CANDIDATES]
    y = dev_df[TARGET].values
    params = _xgb_params_from(best_params)
    params["early_stopping_rounds"] = 60
    oof_sum = np.zeros(len(dev_df))
    oof_cnt = np.zeros(len(dev_df))
    oof_by_repeat, best_iters, fold_scores = [], [], []
    for r in range(CV_REPEATS):
        oof_r = np.full(len(dev_df), np.nan)
        for tr, va in SHARED_FOLDS[r * N_FOLDS:(r + 1) * N_FOLDS]:
            prep = PipelinePreprocessor(features, cat_cols).fit(dev_df.iloc[tr])
            model = xgb.XGBRegressor(**params)
            model.fit(prep.transform_xgb(dev_df.iloc[tr]), y[tr],
                      eval_set=[(prep.transform_xgb(dev_df.iloc[va]), y[va])], verbose=False)
            oof_r[va] = model.predict(prep.transform_xgb(dev_df.iloc[va]))
            best_iters.append(int(model.best_iteration) + 1 if model.best_iteration is not None
                              else XGB_MAX_ROUNDS)
            fold_scores.append(rmse(y[va], oof_r[va]))
        oof_by_repeat.append(oof_r)
        oof_sum += np.nan_to_num(oof_r)
        oof_cnt += (~np.isnan(oof_r)).astype(int)
    oof = oof_sum / oof_cnt
    return {
        "oof": oof,
        "oof_by_repeat": oof_by_repeat,
        "best_iterations": best_iters,
        "cv_rmse": float(np.mean(fold_scores)),
        "cv_se": float(np.std(fold_scores, ddof=1) / np.sqrt(len(fold_scores))),
        "repeat_rmse": [rmse(y, a) for a in oof_by_repeat],
    }

<a id="cell-4-3"></a>

### Cell 4.3 — CatBoost Tuning

Same protocol as Cell 4.2: Optuna on the shared folds, per-fold preprocessing, early stopping
on the held-out fold, tuned loss (RMSE vs `Huber:delta`), native `cat_features` support and
native NaN handling for numeric features. The study is resumable exactly as in Cell 4.2, and
the OOF result is saved to `oof_catboost.json`.


In [ ]:
# Cell 4.3 — CatBoost: resumable Optuna tuning + tuned-parameter OOF collection.
CAT_MAX_ITERS = 4000

def _cat_params_from(best_params):
    p = dict(best_params)
    family = p.pop("loss_family")
    if family == "Huber":
        p["loss_function"] = f"Huber:delta={p.pop('huber_delta'):.4f}"
    else:
        p["loss_function"] = "RMSE"
    p.update(iterations=CAT_MAX_ITERS, random_seed=SEED, verbose=0,
             early_stopping_rounds=100, allow_writing_files=False)
    return p

def tune_catboost(features, n_trials, run_name, run_dir, force_rerun=False):
    """Optuna tuning with a resumable SQLite study; returns best_params."""
    cat_cols = [c for c in features if c in CATEGORICAL_CANDIDATES]
    y = dev_df[TARGET].values

    def objective(trial):
        params = {
            "loss_family": trial.suggest_categorical("loss_family", ["RMSE", "Huber"]),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "depth": trial.suggest_int("depth", 4, 8),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        }
        if params["loss_family"] == "Huber":
            params["huber_delta"] = trial.suggest_float("huber_delta", 0.5, 5.0)
        model = CatBoostRegressor(**_cat_params_from(params), cat_features=cat_cols)
        scores = []
        for tr, va in SHARED_FOLDS:
            prep = PipelinePreprocessor(features, cat_cols).fit(dev_df.iloc[tr])
            model.fit(prep.transform_catboost(dev_df.iloc[tr]), y[tr],
                      eval_set=(prep.transform_catboost(dev_df.iloc[va]), y[va]))
            scores.append(rmse(y[va], model.predict(prep.transform_catboost(dev_df.iloc[va]))))
        trial.set_user_attr("fold_rmse", scores)
        return float(np.mean(scores))

    return _tune_with_resume("catboost", run_dir, objective, n_trials, run_name, features,
                             force_rerun=force_rerun)

def catboost_oof(features, best_params, run_name):
    """Tuned-parameter OOF predictions + per-fold best iteration counts."""
    cat_cols = [c for c in features if c in CATEGORICAL_CANDIDATES]
    y = dev_df[TARGET].values
    params = _cat_params_from(best_params)
    oof_sum = np.zeros(len(dev_df))
    oof_cnt = np.zeros(len(dev_df))
    oof_by_repeat, best_iters, fold_scores = [], [], []
    for r in range(CV_REPEATS):
        oof_r = np.full(len(dev_df), np.nan)
        for tr, va in SHARED_FOLDS[r * N_FOLDS:(r + 1) * N_FOLDS]:
            prep = PipelinePreprocessor(features, cat_cols).fit(dev_df.iloc[tr])
            model = CatBoostRegressor(**params, cat_features=cat_cols)
            model.fit(prep.transform_catboost(dev_df.iloc[tr]), y[tr],
                      eval_set=(prep.transform_catboost(dev_df.iloc[va]), y[va]))
            oof_r[va] = model.predict(prep.transform_catboost(dev_df.iloc[va]))
            best_iters.append(int(model.get_best_iteration()) + 1
                              if model.get_best_iteration() is not None else CAT_MAX_ITERS)
            fold_scores.append(rmse(y[va], oof_r[va]))
        oof_by_repeat.append(oof_r)
        oof_sum += np.nan_to_num(oof_r)
        oof_cnt += (~np.isnan(oof_r)).astype(int)
    oof = oof_sum / oof_cnt
    return {
        "oof": oof,
        "oof_by_repeat": oof_by_repeat,
        "best_iterations": best_iters,
        "cv_rmse": float(np.mean(fold_scores)),
        "cv_se": float(np.std(fold_scores, ddof=1) / np.sqrt(len(fold_scores))),
        "repeat_rmse": [rmse(y, a) for a in oof_by_repeat],
    }

<a id="cell-4-4"></a>

### Cell 4.4 — TabNet Tuning

`TabNetRegressor` with the training-fold target standardisation (mean/std of the training
fold; predictions inverted with the same statistics), `y` shaped `(-1, 1)`, label-encoded
categoricals with embeddings and missing indicators via the Cell 3.2 TabNet view (whose
numeric columns are standardised with training statistics — raw-scale inputs make TabNet's
slowly-adapting BatchNorm running statistics diverge on small data), early stopping on the
held-out fold, GPU when available, and small batch sizes (clamped to the fold size) so each
epoch gives the network enough weight updates.


In [ ]:
# Cell 4.4 — TabNet: resumable Optuna tuning + tuned-parameter OOF collection.
MAX_TABNET_EPOCHS = 25 if FAST_DEBUG else 200
TABNET_PATIENCE   = 10 if FAST_DEBUG else 30

def _fit_tabnet_fold(features, tab_kwargs, lr, batch_size, tr, va, max_epochs, patience,
                     collect_best=False):
    """Fit one TabNet fold with training-fold target standardisation.

    Returns (pred_va_log, best_epoch). Predictions are inverted back to the log scale
    with the SAME training-fold statistics — nothing from the held-out fold is used
    for standardisation.
    """
    cat_cols = [c for c in features if c in CATEGORICAL_CANDIDATES]
    y = dev_df[TARGET].values
    prep = PipelinePreprocessor(features, cat_cols).fit(dev_df.iloc[tr])
    Xtr = prep.transform_tabnet(dev_df.iloc[tr]).values.astype(np.float32)
    Xva = prep.transform_tabnet(dev_df.iloc[va]).values.astype(np.float32)
    cat_idxs, cat_dims = prep.tabnet_cat_info()
    kwargs = dict(tab_kwargs)
    if cat_idxs:
        kwargs.update(cat_idxs=cat_idxs, cat_dims=cat_dims,
                      cat_emb_dim=[min(10, 1 + d // 2) for d in cat_dims])
    mu = float(y[tr].mean())
    sd = float(y[tr].std())
    if sd == 0.0:
        sd = 1.0
    model = TabNetRegressor(seed=SEED, verbose=0, device_name=DEVICE, **kwargs)
    bs = int(min(batch_size, len(tr)))
    vbs = int(max(8, bs // 2))
    model.fit(
        Xtr, ((y[tr] - mu) / sd).reshape(-1, 1),
        eval_set=[(Xva, ((y[va] - mu) / sd).reshape(-1, 1))],
        eval_metric=["rmse"],
        max_epochs=max_epochs, patience=patience,
        batch_size=bs, virtual_batch_size=vbs,
    )
    pred = model.predict(Xva).ravel() * sd + mu
    return pred, int(getattr(model, "best_epoch", max_epochs) or max_epochs)

def tune_tabnet(features, n_trials, run_name, run_dir, force_rerun=False):
    """Optuna tuning with a resumable SQLite study; returns best_params."""
    def objective(trial):
        n_d = trial.suggest_int("n_d", 8, 64, step=8)
        tab_kwargs = {
            "n_d": n_d, "n_a": n_d,
            "n_steps": trial.suggest_int("n_steps", 3, 8),
            "gamma": trial.suggest_float("gamma", 1.0, 2.0),
            "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-2, log=True),
            "momentum": trial.suggest_float("momentum", 0.01, 0.1, log=True),
        }
        lr = trial.suggest_float("lr", 0.005, 0.05, log=True)
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
        scores = []
        for tr, va in SHARED_FOLDS:
            pred, _ = _fit_tabnet_fold(features, tab_kwargs, lr, batch_size,
                                       tr, va, MAX_TABNET_EPOCHS, TABNET_PATIENCE)
            scores.append(rmse(dev_df[TARGET].values[va], pred))
        trial.set_user_attr("fold_rmse", scores)
        return float(np.mean(scores))

    return _tune_with_resume("tabnet", run_dir, objective, n_trials, run_name, features,
                             force_rerun=force_rerun)

def tabnet_oof(features, best_params, run_name):
    """Tuned-parameter OOF predictions + per-fold best epoch counts."""
    n_d = int(best_params["n_d"])
    tab_kwargs = {
        "n_d": n_d, "n_a": n_d,
        "n_steps": int(best_params["n_steps"]),
        "gamma": float(best_params["gamma"]),
        "lambda_sparse": float(best_params["lambda_sparse"]),
        "momentum": float(best_params["momentum"]),
    }
    lr = float(best_params["lr"])
    batch_size = int(best_params["batch_size"])
    y = dev_df[TARGET].values
    oof_sum = np.zeros(len(dev_df))
    oof_cnt = np.zeros(len(dev_df))
    oof_by_repeat, best_epochs, fold_scores = [], [], []
    for r in range(CV_REPEATS):
        oof_r = np.full(len(dev_df), np.nan)
        for tr, va in SHARED_FOLDS[r * N_FOLDS:(r + 1) * N_FOLDS]:
            pred, best_epoch = _fit_tabnet_fold(
                features, tab_kwargs, lr, batch_size, tr, va,
                MAX_TABNET_EPOCHS, TABNET_PATIENCE, collect_best=True)
            oof_r[va] = pred
            best_epochs.append(best_epoch)
            fold_scores.append(rmse(y[va], pred))
        oof_by_repeat.append(oof_r)
        oof_sum += np.nan_to_num(oof_r)
        oof_cnt += (~np.isnan(oof_r)).astype(int)
    oof = oof_sum / oof_cnt
    return {
        "oof": oof,
        "oof_by_repeat": oof_by_repeat,
        "best_epochs": best_epochs,
        "cv_rmse": float(np.mean(fold_scores)),
        "cv_se": float(np.std(fold_scores, ddof=1) / np.sqrt(len(fold_scores))),
        "repeat_rmse": [rmse(y, a) for a in oof_by_repeat],
        "lr": lr, "batch_size": batch_size, "tab_kwargs": tab_kwargs,
    }

phase_timer_print("Phase 4 — base-model machinery (tuning runs inside Phase 6)")

<a id="phase-5"></a>

## Phase 5 — Ensemble

Out-of-fold predictions from the three tuned models are combined with non-negative weights
that sum to 1 (SLSQP on the development-set OOF matrix — weights are a decision, so they are
fitted on CV predictions, never on test rows). A simple average is reported alongside. Each
model is then re-fitted on **all** development rows with the iteration counts (rounds /
iterations / epochs) found during cross-validation, ready for the single test-set evaluation.


<a id="cell-5-1"></a>

### Cell 5.1 — Ensemble Weights

Weights are constrained to the simplex (`0 <= w <= 1`, `sum w = 1`) and minimise the weighted
OOF prediction's RMSE on the development set. `MODEL_ORDER` fixes the column order of the OOF
matrix everywhere in the notebook.


In [ ]:
# Cell 5.1 — Non-negative simplex weights on development-set OOF predictions.
phase_timer_start()   # Phase 5 starts here; Cell 5.2 prints its elapsed time.
MODEL_ORDER = ["xgb", "catboost", "tabnet"]

def fit_ensemble_weights(oof_matrix, y_true):
    """SLSQP: minimise RMSE of the weighted OOF prediction, 0 <= w <= 1, sum(w) == 1."""
    k = oof_matrix.shape[1]

    def objective(w):
        return rmse(y_true, oof_matrix @ w)

    res = minimize(objective, x0=np.full(k, 1.0 / k), method="SLSQP",
                   bounds=[(0.0, 1.0)] * k,
                   constraints=[{"type": "eq", "fun": lambda w: float(np.sum(w) - 1.0)}],
                   options={"maxiter": 1000, "ftol": 1e-12})
    w = np.clip(res.x, 0.0, 1.0)
    return w / w.sum()

print("PHASE 5.1 — ensemble weight fitter ready (non-negative, sum to 1); "
      f"model order: {MODEL_ORDER}")

<a id="cell-5-2"></a>

### Cell 5.2 — Refit on Development Data

Re-fits each tuned model on all development rows with fixed iteration budgets — median of the
per-fold best round counts for XGBoost / CatBoost, median best epoch for TabNet (early
stopping disabled by setting patience to the epoch budget; the development set itself is used
only as the monitored set). TabNet's target is standardised with development statistics, which
are stored for inference. The same routine returns the development-fitted preprocessor that
Phase 6 applies to the test set — the test rows are never used to fit anything.

The whole stage (weights + refit + preprocessor) is resumable: the models, the ensemble
weights, the preprocessor dictionary and TabNet's target statistics are saved under
`models/runs/<run>/` (`refit_*` files, `ensemble_weights.json`, `preprocessing.json`) and
re-loaded — the stage is skipped — when those files already match the current configuration
signature.


In [ ]:
# Cell 5.2 — Refit every model on all development rows with the CV iteration counts.
def refit_models(features, best_params, oof_results, run_name):
    """best_params: {'xgb': tune_xgb result, 'catboost': tune_catboost result,
                     'tabnet': tune_tabnet result}.
    oof_results:  {'xgb': xgb_oof result, 'catboost': catboost_oof result,
                   'tabnet': tabnet_oof result} (iteration counts + rebuilt TabNet kwargs).
    Returns (prep, models, tabnet_target_stats)."""
    cat_cols = [c for c in features if c in CATEGORICAL_CANDIDATES]
    y = dev_df[TARGET].values
    prep = PipelinePreprocessor(features, cat_cols).fit(dev_df)
    models = {}

    # XGBoost — fixed round count = median of the fold best-iteration counts.
    n_rounds = int(np.ceil(np.median(oof_results["xgb"]["best_iterations"])))
    xgb_params = _xgb_params_from(best_params["xgb"])
    xgb_params.pop("early_stopping_rounds", None)
    xgb_params["n_estimators"] = n_rounds
    models["xgb"] = xgb.XGBRegressor(**xgb_params)
    models["xgb"].fit(prep.transform_xgb(dev_df), y, verbose=False)

    # CatBoost — fixed iteration count = median of the fold best-iteration counts.
    n_iters = int(np.ceil(np.median(oof_results["catboost"]["best_iterations"])))
    cat_params = _cat_params_from(best_params["catboost"])
    cat_params.pop("early_stopping_rounds", None)
    cat_params["iterations"] = n_iters
    models["catboost"] = CatBoostRegressor(**cat_params, cat_features=cat_cols)
    models["catboost"].fit(prep.transform_catboost(dev_df), y)

    # TabNet — fixed epoch budget = median of the fold best epochs; patience disables
    # early stopping; the development set is the monitored set (statistics only from dev).
    n_epochs = max(10, int(np.ceil(np.median(oof_results["tabnet"]["best_epochs"]))))
    mu = float(y.mean())
    sd = float(y.std())
    if sd == 0.0:
        sd = 1.0
    tab_kwargs = dict(oof_results["tabnet"]["tab_kwargs"])
    cat_idxs, cat_dims = prep.tabnet_cat_info()
    if cat_idxs:
        tab_kwargs.update(cat_idxs=cat_idxs, cat_dims=cat_dims,
                          cat_emb_dim=[min(10, 1 + d // 2) for d in cat_dims])
    X_dev_tab = prep.transform_tabnet(dev_df).values.astype(np.float32)
    bs = int(min(oof_results["tabnet"]["batch_size"], len(dev_df)))
    vbs = int(max(8, bs // 2))
    models["tabnet"] = TabNetRegressor(seed=SEED, verbose=0, device_name=DEVICE, **tab_kwargs)
    models["tabnet"].fit(
        X_dev_tab, ((y - mu) / sd).reshape(-1, 1),
        eval_set=[(X_dev_tab, ((y - mu) / sd).reshape(-1, 1))],
        eval_metric=["rmse"],
        max_epochs=n_epochs, patience=n_epochs,      # train the full budget
        batch_size=bs, virtual_batch_size=vbs,
    )
    tabnet_target_stats = {"tabnet_target_mean": mu, "tabnet_target_std": sd}

    print(f"  [{run_name}] refit on {len(dev_df)} dev rows — xgb rounds={n_rounds}, "
          f"catboost iters={n_iters}, tabnet epochs={n_epochs}")
    return prep, models, tabnet_target_stats

def _refit_with_resume(features, best, oofr, run_name, run_dir, force_rerun=False):
    """Ensemble weights + development refit, saved per run and skipped when on file.

    Files (all under run_dir): refit_xgb_model.json, refit_catboost_model.cbm,
    refit_tabnet_model.zip, ensemble_weights.json, preprocessing.json. The signature
    chains the OOF signatures, so a recomputed OOF stage invalidates a stale refit.
    """
    sig = _stage_signature({"kind": "refit", "features": sorted(features),
                            "best_params": best,
                            "oof_signatures": {k: oofr[k].get("signature")
                                               for k in MODEL_ORDER},
                            "n_folds": N_FOLDS, "repeats": CV_REPEATS, "seed": SEED})
    files = ["refit_xgb_model.json", "refit_catboost_model.cbm", "refit_tabnet_model.zip",
             "ensemble_weights.json", "preprocessing.json"]
    if not force_rerun and all(os.path.exists(os.path.join(run_dir, f)) for f in files):
        with open(os.path.join(run_dir, "preprocessing.json")) as f:
            saved_pre = json.load(f)
        if saved_pre.get("signature") == sig:
            with open(os.path.join(run_dir, "ensemble_weights.json")) as f:
                w_saved = json.load(f)
            prep = PipelinePreprocessor.from_dict(saved_pre["preprocessor"])
            tabnet_target = saved_pre["tabnet_target"]
            weights = np.asarray(w_saved["weights"], float)
            models = {"xgb": xgb.XGBRegressor(), "catboost": CatBoostRegressor(),
                      "tabnet": TabNetRegressor()}
            models["xgb"].load_model(os.path.join(run_dir, "refit_xgb_model.json"))
            # load_model() restores the booster but NOT the sklearn-side training
            # parameters — re-apply enable_categorical so category-dtype frames keep
            # working (in particular for shap.TreeExplainer, which reads the wrapper).
            models["xgb"].set_params(enable_categorical=True, tree_method="hist")
            models["catboost"].load_model(os.path.join(run_dir, "refit_catboost_model.cbm"))
            models["tabnet"].load_model(os.path.join(run_dir, "refit_tabnet_model.zip"))
            print(f"  [{run_name}] refit stage found on file — SKIPPED (5 artefacts re-loaded)")
            return prep, models, tabnet_target, weights
    t_refit = time.time()
    oof_matrix = np.column_stack([oofr[k]["oof"] for k in MODEL_ORDER])
    weights = fit_ensemble_weights(oof_matrix, dev_df[TARGET].values)
    prep, models, tabnet_target = refit_models(features, best, oofr, run_name)
    models["xgb"].save_model(os.path.join(run_dir, "refit_xgb_model.json"))
    models["catboost"].save_model(os.path.join(run_dir, "refit_catboost_model.cbm"))
    models["tabnet"].save_model(os.path.join(run_dir, "refit_tabnet_model"))  # .zip appended
    with open(os.path.join(run_dir, "ensemble_weights.json"), "w") as f:
        json.dump({"model_order": MODEL_ORDER,
                   "weights": [float(w) for w in weights],
                   "signature": sig,
                   "fitted_on": "development-set OOF predictions"}, f, indent=2)
    with open(os.path.join(run_dir, "preprocessing.json"), "w") as f:
        json.dump({"preprocessor": prep.to_dict(), "tabnet_target": tabnet_target,
                   "train_median_log": TRAIN_MEDIAN_LOG, "signature": sig}, f, indent=2)
    print(f"  [{run_name}] refit stage computed and saved ({time.time() - t_refit:.1f}s)")
    return prep, models, tabnet_target, weights

print("PHASE 5.2 — refit routine + resumable refit stage ready.")
phase_timer_print("Phase 5 — ensemble machinery (refits run inside Phase 6)")

<a id="phase-6"></a>

## Phase 6 — Evaluation

Cells 6.1–6.2 define the evaluation tools: bootstrap confidence intervals for the test
metrics, the outlier-trimmed sensitivity subset (test rows inside the development target's
1st–99th percentile), and a county-grouped `GroupKFold` CV score reported next to the
standard CV score. Cell 6.3 packages the modelling into one reusable, resumable routine —
tune on the shared folds, fit ensemble weights on out-of-fold predictions, refit on all
development rows — which is then called with identical budgets: **Run A** (the 9 base
features), **Run B** (`SELECTED_FEATURES`) and, when `CONFIG.RUN_ALTERNATIVE` is True,
**Run C** (`ALTERNATIVE_FEATURES`, the minimum-CV-error set). Each run reads the test set
exactly once, in its final evaluation cell, and only to report numbers. Nothing on the test
set is standardised except with training (development) statistics. Cell 6.6 puts every
registered run into one comparison table and states plainly when the ensemble does not beat
the best single model.


<a id="cell-6-1"></a>

### Cell 6.1 — Test Evaluation Helpers

95% percentile bootstrap confidence intervals (resampling test rows with a fixed RNG) for the
log-scale test metrics.


In [ ]:
# Cell 6.1 — Bootstrap confidence intervals for the test metrics.
phase_timer_start()   # Phase 6 starts here; Cell 6.6 prints its elapsed time.
N_BOOTSTRAP = 200 if FAST_DEBUG else 2000

def bootstrap_test_ci(y_true, y_pred, n_boot=N_BOOTSTRAP, seed=SEED, alpha=0.05):
    """95% percentile bootstrap CIs for R2 / RMSE / MAE, resampling the test rows."""
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    n = len(y_true)
    draws = {"R2": [], "RMSE": [], "MAE": []}
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        m = regression_metrics(y_true[idx], y_pred[idx])
        for k in draws:
            draws[k].append(m[k])
    return {k: (float(np.nanpercentile(v, 100 * alpha / 2)),
                float(np.nanpercentile(v, 100 * (1 - alpha / 2))))
            for k, v in draws.items()}

print(f"PHASE 6.1 — bootstrap helper ready ({N_BOOTSTRAP} resamples, seed {SEED}).")

<a id="cell-6-2"></a>

### Cell 6.2 — Sensitivity and County CV

`county_group_cv` re-runs cross-validation with the tuned parameters under the county-grouped
folds (whole counties held out together) so a geographic-honesty score can sit next to the
standard CV score; the ensemble uses the weights already fitted on the standard OOF matrix.


In [ ]:
# Cell 6.2 — Trimmed-subset sensitivity + county-grouped CV with tuned parameters.
def county_group_cv(run):
    """GroupKFold-by-county CV RMSE per model (+ weighted ensemble) with the run's params."""
    features = run["features"]
    best, w = run["best_params"], run["weights"]
    y = dev_df[TARGET].values
    oof_group = {}

    xgb_params = _xgb_params_from(best["xgb"])
    xgb_params["early_stopping_rounds"] = 60
    cat_params = _cat_params_from(best["catboost"])
    tab_kwargs = dict(run["oof_results"]["tabnet"]["tab_kwargs"])
    lr = run["oof_results"]["tabnet"]["lr"]
    batch_size = run["oof_results"]["tabnet"]["batch_size"]

    for model_key in MODEL_ORDER:
        oof = np.full(len(dev_df), np.nan)
        for tr, va in GROUP_FOLDS:
            if model_key == "xgb":
                prep = PipelinePreprocessor(features, [c for c in features
                                                       if c in CATEGORICAL_CANDIDATES]
                                            ).fit(dev_df.iloc[tr])
                model = xgb.XGBRegressor(**xgb_params)
                model.fit(prep.transform_xgb(dev_df.iloc[tr]), y[tr],
                          eval_set=[(prep.transform_xgb(dev_df.iloc[va]), y[va])], verbose=False)
                oof[va] = model.predict(prep.transform_xgb(dev_df.iloc[va]))
            elif model_key == "catboost":
                prep = PipelinePreprocessor(features, [c for c in features
                                                       if c in CATEGORICAL_CANDIDATES]
                                            ).fit(dev_df.iloc[tr])
                model = CatBoostRegressor(**cat_params,
                                          cat_features=[c for c in features
                                                        if c in CATEGORICAL_CANDIDATES])
                model.fit(prep.transform_catboost(dev_df.iloc[tr]), y[tr],
                          eval_set=(prep.transform_catboost(dev_df.iloc[va]), y[va]))
                oof[va] = model.predict(prep.transform_catboost(dev_df.iloc[va]))
            else:
                pred, _ = _fit_tabnet_fold(features, tab_kwargs, lr, batch_size,
                                           tr, va, MAX_TABNET_EPOCHS, TABNET_PATIENCE)
                oof[va] = pred
        oof_group[model_key] = oof
    ens = sum(w[i] * oof_group[k] for i, k in enumerate(MODEL_ORDER))
    scores = {k: rmse(y, oof_group[k]) for k in MODEL_ORDER}
    scores["weighted_ensemble"] = rmse(y, ens)
    return scores

print("PHASE 6.2 — sensitivity + county-grouped CV helpers ready.")

<a id="cell-6-3"></a>

### Cell 6.3 — Reusable Run Routine

`run_modelling_pipeline` is the Phase 4–6 routine, executed once per run with three resumable
stages: **(1) Optuna tuning** of the three models on the shared folds (SQLite studies,
`load_if_exists=True`, identical budgets for every run), **(2) out-of-fold prediction**
collection per model (saved to `oof_*.json`), and **(3) ensemble weights + refit** on all
development rows (models and preprocessing saved under the run directory). Finished stages
are skipped when their files exist (honouring `FORCE_RERUN`), and every stage prints its
elapsed time. The CV summary is recomputed from the (loaded or fresh) OOF predictions and
weights, so it is always consistent with what later phases see. Called once per registered
run.


In [ ]:
# Cell 6.3 — Phases 4-6 as one reusable, resumable routine.
def run_modelling_pipeline(run_name, run_slug, features):
    """Tune -> OOF -> ensemble weights + refit. Resumable; returns everything later
    phases need. Stages whose saved files already exist (and match the current
    configuration signature) are skipped; FORCE_RERUN overrides."""
    t0 = time.time()
    features = list(features)
    _missing = [c for c in features if c not in dev_df.columns]
    if _missing:
        raise ValueError(f"[{run_name}] features missing from the data: {_missing}")
    run_dir = os.path.join(RUN_DIR, run_slug)
    os.makedirs(run_dir, exist_ok=True)
    print(f"===== {run_name} — {len(features)} features: {features}")
    print(f"      stage files under: {run_dir}")

    # Stage 1/3 — Optuna tuning (SQLite studies under the run directory).
    t_stage = time.time()
    best = {
        "xgb":      tune_xgb(features, N_TRIALS_XGB, run_name, run_dir, FORCE_RERUN),
        "catboost": tune_catboost(features, N_TRIALS_CATBOOST, run_name, run_dir, FORCE_RERUN),
        "tabnet":   tune_tabnet(features, N_TRIALS_TABNET, run_name, run_dir, FORCE_RERUN),
    }
    print(f"  [{run_name}] stage 1/3 (tuning) done in {time.time() - t_stage:.1f}s")

    # Stage 2/3 — out-of-fold predictions per model (saved to oof_<model>.json).
    t_stage = time.time()
    oofr = {
        "xgb":      _oof_with_resume("xgb", xgb_oof, features, best["xgb"],
                                    run_name, run_dir, FORCE_RERUN),
        "catboost": _oof_with_resume("catboost", catboost_oof, features, best["catboost"],
                                    run_name, run_dir, FORCE_RERUN),
        "tabnet":   _oof_with_resume("tabnet", tabnet_oof, features, best["tabnet"],
                                    run_name, run_dir, FORCE_RERUN),
    }
    print(f"  [{run_name}] stage 2/3 (out-of-fold) done in {time.time() - t_stage:.1f}s")

    # Stage 3/3 — simplex ensemble weights + refit on all development rows.
    t_stage = time.time()
    prep, models, tabnet_target, weights = _refit_with_resume(
        features, best, oofr, run_name, run_dir, FORCE_RERUN)
    print(f"  [{run_name}] stage 3/3 (weights + refit) done in {time.time() - t_stage:.1f}s")

    # CV summary — recomputed from the (fresh or loaded) OOF predictions and weights.
    oof_matrix = np.column_stack([oofr[k]["oof"] for k in MODEL_ORDER])
    y_dev = dev_df[TARGET].values
    ens_oof = oof_matrix @ weights
    avg_oof = oof_matrix.mean(axis=1)
    ens_repeat, avg_repeat = [], []
    for r in range(CV_REPEATS):
        ens_r = sum(weights[i] * oofr[k]["oof_by_repeat"][r] for i, k in enumerate(MODEL_ORDER))
        avg_r = np.mean([oofr[k]["oof_by_repeat"][r] for k in MODEL_ORDER], axis=0)
        ens_repeat.append(rmse(y_dev, ens_r))
        avg_repeat.append(rmse(y_dev, avg_r))
    cv = {k: (oofr[k]["cv_rmse"], oofr[k]["cv_se"]) for k in MODEL_ORDER}
    cv["simple_average"] = (float(np.mean(avg_repeat)),
                            float(np.std(avg_repeat, ddof=1) / np.sqrt(len(avg_repeat))))
    cv["weighted_ensemble"] = (float(np.mean(ens_repeat)),
                               float(np.std(ens_repeat, ddof=1) / np.sqrt(len(ens_repeat))))

    print(f"  [{run_name}] CV summary (development set, repeated K-fold):")
    for k in MODEL_ORDER + ["simple_average", "weighted_ensemble"]:
        print(f"    {k:18s} RMSE {cv[k][0]:.4f} (se {cv[k][1]:.4f})")
    print(f"    ensemble weights     {dict(zip(MODEL_ORDER, np.round(weights, 4)))}")
    print(f"    dev OOF RMSE of weighted ensemble {rmse(y_dev, ens_oof):.4f} | "
          f"simple average {rmse(y_dev, avg_oof):.4f} | run total {time.time() - t0:.1f}s")

    return {
        "run_name": run_name,
        "run_slug": run_slug,
        "run_dir": run_dir,
        "features": features,
        "best_params": best,
        "oof_results": oofr,
        "oof_matrix": oof_matrix,
        "weights": weights,
        "cv": cv,
        "prep": prep,
        "models": models,
        "tabnet_target": tabnet_target,
        "seconds": time.time() - t0,
    }

print("PHASE 6.3 — run_modelling_pipeline ready (called once per run below).")

<a id="cell-6-4"></a>

### Cell 6.4 — Run A

Run A uses the original 9 base features on the full target-filtered frame. The test set is
read **here** — once for this run — and only to report final numbers. Predictions use the
development-fitted preprocessor and models; TabNet's standardized output is inverted with
development statistics only. The run's baselines (training median, OLS re-fitted on the
development set with the run's features, median imputation + missing indicators as in
Cell 2.1) are evaluated on the same rows so every row of the report is like-for-like.


In [ ]:
# Cell 6.4 — Run A (9 base features): routine + THE single test read for this run.
REPORT_ORDER = ["median_baseline", "ols"] + MODEL_ORDER + ["simple_average", "weighted_ensemble"]

def evaluate_run_on_test(run, test_frame):
    """Final test evaluation. Standardisation/encoding only via the dev-fitted preprocessor."""
    prep, models, w = run["prep"], run["models"], run["weights"]
    tab_mu = run["tabnet_target"]["tabnet_target_mean"]
    tab_sd = run["tabnet_target"]["tabnet_target_std"]
    preds = {
        "xgb": models["xgb"].predict(prep.transform_xgb(test_frame)),
        "catboost": models["catboost"].predict(prep.transform_catboost(test_frame)),
        "tabnet": (models["tabnet"]
                   .predict(prep.transform_tabnet(test_frame).values.astype(np.float32))
                   .ravel() * tab_sd + tab_mu),
    }
    preds["simple_average"] = np.mean([preds[k] for k in MODEL_ORDER], axis=0)
    preds["weighted_ensemble"] = sum(w[i] * preds[k] for i, k in enumerate(MODEL_ORDER))
    ols_run = fit_ols(dev_df, run["features"])
    preds["ols"] = ols_run.predict(test_frame[run["features"]])
    preds["median_baseline"] = np.full(len(test_frame), TRAIN_MEDIAN_LOG)

    y_test = test_frame[TARGET].values
    out = {"y_true": y_test, "preds": preds, "metrics": {}, "ci": {}}
    for k in REPORT_ORDER:
        out["metrics"][k] = regression_metrics(y_test, preds[k])
        out["ci"][k] = bootstrap_test_ci(y_test, preds[k])
        print(f"  {k:18s} {format_metrics(out['metrics'][k])}")
        print(f"  {'':18s} 95% CI  R2 [{out['ci'][k]['R2'][0]:.4f}, {out['ci'][k]['R2'][1]:.4f}] "
              f"RMSE [{out['ci'][k]['RMSE'][0]:.4f}, {out['ci'][k]['RMSE'][1]:.4f}] "
              f"MAE [{out['ci'][k]['MAE'][0]:.4f}, {out['ci'][k]['MAE'][1]:.4f}]")

    lo, hi = np.percentile(dev_df[TARGET].values, [1, 99])
    mask = (y_test >= lo) & (y_test <= hi)
    out["trimmed"] = {
        "lo": float(lo), "hi": float(hi), "n_kept": int(mask.sum()),
        "metrics": regression_metrics(y_test[mask], preds["weighted_ensemble"][mask]),
    }
    print(f"  Sensitivity — test rows inside the development target's 1st-99th percentile "
          f"({out['trimmed']['n_kept']} of {len(y_test)} rows): "
          f"{format_metrics(out['trimmed']['metrics'])}")

    out["group_cv"] = county_group_cv(run)
    print("  County-grouped CV next to the standard CV (RMSE, development set):")
    for k in MODEL_ORDER + ["weighted_ensemble"]:
        print(f"    {k:18s} grouped {out['group_cv'][k]:.4f} | standard {run['cv'][k][0]:.4f}")
    return out

RUN_A = run_modelling_pipeline("Run A (9 base features)", "run_a", BASE_FEATURES)
print("\nPHASE 6.4 — RUN A, final test evaluation (test set read once for this run):")
RUN_A_TEST = evaluate_run_on_test(RUN_A, test_df)

<a id="cell-6-5"></a>

### Cell 6.5 — Run B and Optional Run C

Run B uses `SELECTED_FEATURES` from Cell 3.8 (or hard-coded in CONFIG). Same folds, same
tuning budgets, same evaluation protocol as Run A; the test set is read once, here. When
`CONFIG.RUN_ALTERNATIVE` is True, the minimum-CV-error set `ALTERNATIVE_FEATURES` is also
run (as **Run C**) — unless it duplicates a feature set already run, in which case the
notebook says so and skips it. Every executed run is registered in `RUNS`, which the later
phases iterate over.


In [ ]:
# Cell 6.5 — Run B (selected features) + optional Run C (minimum-CV-error features).
if not SELECTED_FEATURES:
    raise ValueError(
        "SELECTED_FEATURES is empty — run Phase 3 (Cells 3.1-3.8) first, or hard-code "
        "SELECTED_FEATURES in the CONFIG cell (1.6)."
    )
if sorted(SELECTED_FEATURES) == sorted(BASE_FEATURES):
    print("Note: Run B's feature set equals Run A's 9 base features — the runs will mirror "
          "each other up to tuning noise.")

RUN_B = run_modelling_pipeline("Run B (selected features)", "run_b", SELECTED_FEATURES)
print("\nPHASE 6.5 — RUN B, final test evaluation (test set read once for this run):")
RUN_B_TEST = evaluate_run_on_test(RUN_B, test_df)

RUNS = [("Run A", RUN_A, RUN_A_TEST), ("Run B", RUN_B, RUN_B_TEST)]

# Optional Run C — the minimum-CV-error feature set, only when CONFIG.RUN_ALTERNATIVE.
RUN_C = RUN_C_TEST = None
if RUN_ALTERNATIVE:
    _alt_sorted = sorted(ALTERNATIVE_FEATURES)
    if _alt_sorted == sorted(SELECTED_FEATURES):
        print("\nRUN_ALTERNATIVE is True but the alternative set equals the selected set "
              "— Run C skipped (it would mirror Run B).")
    elif _alt_sorted == sorted(BASE_FEATURES):
        print("\nRUN_ALTERNATIVE is True but the alternative set equals the 9 base "
              "features — Run C skipped (it would mirror Run A).")
    else:
        RUN_C = run_modelling_pipeline("Run C (alternative minimum-CV features)",
                                       "run_c", ALTERNATIVE_FEATURES)
        print("\nRUN C (optional), final test evaluation (test set read once for this run):")
        RUN_C_TEST = evaluate_run_on_test(RUN_C, test_df)
        RUNS.append(("Run C", RUN_C, RUN_C_TEST))

print(f"\nRuns registered for the comparison, charts and production choice: "
      f"{[_lbl for _lbl, _, _ in RUNS]}")

<a id="cell-6-6"></a>

### Cell 6.6 — Comparison Table

One table: every registered run (baseline, each model, simple average, weighted ensemble).
The archived OLS / MLP test numbers are deliberately absent — they were produced on the
22 train / 5 validation / 5 test row splits of the archived 32-row frame and are not
comparable; see the note after the CONFIG cell. Followed by a plain statement of whether
each run's ensemble beats its best single model, and the CV-vs-county-grouped CV summary.


In [ ]:
# Cell 6.6 — Comparison table across every registered run.
_rows = []
for _run_label, _run, _run_test in RUNS:
    for _k in REPORT_ORDER:
        _m = _run_test["metrics"][_k]
        _rows.append({"run": _run_label, "model": _k,
                      "R2": round(_m["R2"], 4), "RMSE": round(_m["RMSE"], 4),
                      "MAE": round(_m["MAE"], 4),
                      "MedAPE_%": round(_m["MedAPE_%"], 2),
                      "MAE_KSh_per_acre": round(_m["MAE_KSh_per_acre"], 0)})

comparison_table = pd.DataFrame(_rows)
print("PHASE 6.6 — comparison table (log-scale R2/RMSE/MAE; money metrics on the median basis)")
print(comparison_table.to_string(index=False))

print("\nEnsemble vs best single model (test RMSE, log scale):")
for _run_label, _run, _run_test in RUNS:
    _single = {k: _run_test["metrics"][k]["RMSE"] for k in MODEL_ORDER}
    _best_k = min(_single, key=_single.get)
    _ens = _run_test["metrics"]["weighted_ensemble"]["RMSE"]
    if _ens > _single[_best_k]:
        print(f"  {_run_label}: the weighted ensemble does NOT beat the best single model "
              f"({_best_k}, RMSE {_single[_best_k]:.4f} vs ensemble {_ens:.4f}). "
              "Stated plainly — the single model is the better choice on this split.")
    else:
        print(f"  {_run_label}: the weighted ensemble beats every single model "
              f"(ensemble RMSE {_ens:.4f} vs best single {_best_k} {_single[_best_k]:.4f}).")

print("\nCV summary next to county-grouped CV (RMSE, development set):")
for _run_label, _run, _run_test in RUNS:
    for _k in MODEL_ORDER + ["weighted_ensemble"]:
        print(f"  {_run_label} {_k:18s} standard {_run['cv'][_k][0]:.4f} | "
              f"county-grouped {_run_test['group_cv'][_k]:.4f}")
phase_timer_print("Phase 6 — runs and evaluation")

<a id="phase-7"></a>

## Phase 7 — Interpretation

Charts for every registered run, saved to `OUTPUTS`: predicted vs actual and residual plots
(log scale), a model comparison chart, XGBoost feature importance for every run, and a SHAP
summary for the run with the best CV ensemble RMSE (a development-set choice — the test set
plays no role here). Every number quoted in this phase is computed and printed at run time.


<a id="cell-7-1"></a>

### Cell 7.1 — Predicted vs Actual and Residuals

Weighted-ensemble predictions of every registered run on the test set, log scale. The identity
line is the reference; residuals are plotted against predicted values with a zero line.


In [ ]:
# Cell 7.1 — Predicted vs actual + residuals for every registered run (test set, log scale).
phase_timer_start()   # Phase 7 starts here; Cell 7.4 prints its elapsed time.
_n_runs = len(RUNS)
fig_pred, _axes_pred = plt.subplots(1, _n_runs, figsize=(5.5 * _n_runs, 4.8),
                                    constrained_layout=True)
fig_res, _axes_res = plt.subplots(1, _n_runs, figsize=(5.5 * _n_runs, 4.8),
                                  constrained_layout=True)
for _j, (_run_label, _run, _run_test) in enumerate(RUNS):
    y_true = _run_test["y_true"]
    y_pred = _run_test["preds"]["weighted_ensemble"]
    _lo = float(min(y_true.min(), y_pred.min()))
    _hi = float(max(y_true.max(), y_pred.max()))
    ax = _axes_pred[_j]
    ax.scatter(y_true, y_pred, s=28, alpha=0.75, edgecolor="k", linewidth=0.3)
    ax.plot([_lo, _hi], [_lo, _hi], "r--", lw=1.2, label="identity")
    ax.set_xlabel("actual log_price_per_acre")
    ax.set_ylabel("predicted log_price_per_acre")
    ax.set_title(f"{_run_label} — weighted ensemble (test)")
    ax.legend()
    resid = y_pred - y_true
    ax = _axes_res[_j]
    ax.scatter(y_pred, resid, s=28, alpha=0.75, edgecolor="k", linewidth=0.3)
    ax.axhline(0.0, color="r", ls="--", lw=1.2)
    ax.set_xlabel("predicted log_price_per_acre")
    ax.set_ylabel("residual (predicted - actual)")
    ax.set_title(f"{_run_label} — residuals (test)")
    print(f"{_run_label}: test R2 "
          f"{_run_test['metrics']['weighted_ensemble']['R2']:.4f}, "
          f"mean residual {resid.mean():+.4f}, residual std {resid.std(ddof=1):.4f}")
fig_pred.savefig(f"{OUTPUTS}/pred_vs_actual.png", dpi=150)
fig_res.savefig(f"{OUTPUTS}/residuals.png", dpi=150)
plt.show()
print(f"PHASE 7.1 — charts saved to {OUTPUTS}/pred_vs_actual.png and residuals.png")

<a id="cell-7-2"></a>

### Cell 7.2 — Model Comparison Chart

Test-set R2 / RMSE / MAE for every registered run's models, simple averages and ensembles,
side by side.


In [ ]:
# Cell 7.2 — Model comparison chart (test metrics, both runs).
_keys = REPORT_ORDER
_fig, _axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
for _ax, (_metric, _title) in zip(_axes, [("R2", "R2 (log scale)"),
                                          ("RMSE", "RMSE (log scale)"),
                                          ("MAE", "MAE (log scale)")]):
    _x = np.arange(len(_keys))
    _bar_w = 0.8 / len(RUNS)
    for _j, (_run_label, _run, _run_test) in enumerate(RUNS):
        _vals = [_run_test["metrics"][k][_metric] for k in _keys]
        _ax.bar(_x - 0.4 + _bar_w * (_j + 0.5), _vals, width=_bar_w,
                label=_run_label, alpha=0.85)
    _ax.set_xticks(_x)
    _ax.set_xticklabels(_keys, rotation=45, ha="right", fontsize=8)
    _ax.set_title(_title)
    _ax.legend(fontsize=8)
plt.savefig(f"{OUTPUTS}/model_comparison.png", dpi=150)
plt.show()
print(f"PHASE 7.2 — chart saved to {OUTPUTS}/model_comparison.png")

<a id="cell-7-3"></a>

### Cell 7.3 — Feature Importance

XGBoost gain importance of every registered run's final (development-fitted) models.


In [ ]:
# Cell 7.3 — XGBoost feature importance (gain) for both runs.
_fig, _axes = plt.subplots(1, len(RUNS),
                           figsize=(6.5 * len(RUNS), max(4, 0.4 * len(CANDIDATE_POOL))),
                           constrained_layout=True)
for _ax, (_run_label, _run, _run_test) in zip(_axes, RUNS):
    _imp = _run["models"]["xgb"].feature_importances_
    _names = _run["features"]
    _order = np.argsort(_imp)
    _ax.barh(np.array(_names)[_order], _imp[_order], alpha=0.85)
    _ax.set_title(f"{_run_label} — XGBoost gain importance")
    _ax.set_xlabel("gain share")
plt.savefig(f"{OUTPUTS}/feature_importance.png", dpi=150)
plt.show()
print(f"PHASE 7.3 — chart saved to {OUTPUTS}/feature_importance.png")

<a id="cell-7-4"></a>

### Cell 7.4 — SHAP Summary

TreeSHAP beeswarm of the XGBoost model of the run with the best CV ensemble RMSE
(development rows — a decision made without the test set). The chosen run and its mean-|SHAP|
ranking are printed at run time.


In [ ]:
# Cell 7.4 — SHAP summary for the best-CV run's XGBoost (development set).
SHAP_RUN = min(RUNS, key=lambda _t: _t[1]["cv"]["weighted_ensemble"][0])[1]
print(f"SHAP run: {SHAP_RUN['run_name']} "
      f"(CV ensemble RMSE {SHAP_RUN['cv']['weighted_ensemble'][0]:.4f})")

X_shap = SHAP_RUN["prep"].transform_xgb(dev_df)
_explainer = shap.TreeExplainer(SHAP_RUN["models"]["xgb"])
_shap_values = np.asarray(_explainer.shap_values(X_shap))

plt.figure()
shap.summary_plot(_shap_values, X_shap, show=False, max_display=20)
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"PHASE 7.4 — SHAP summary for {SHAP_RUN['run_name']}; chart saved to "
      f"{OUTPUTS}/shap_summary.png")
print("Mean |SHAP| ranking (development set):")
for _c, _v in sorted(zip(X_shap.columns, np.abs(_shap_values).mean(axis=0)),
                     key=lambda kv: -kv[1]):
    print(f"  {_c:28s} {_v:.4f}")
phase_timer_print("Phase 7 — interpretation")

<a id="phase-8"></a>

## Phase 8 — Save for Backend

Chooses the production run on development-set evidence only — better CV ensemble RMSE wins;
among runs within one pooled standard error of the best, the run with fewer features wins
(simpler to serve). Saves the three models plus every artefact an inference service needs
into `models/ensemble/`, and defines `predict_price_per_acre`, the documented single-point
entry that returns KSh per acre (and the total price when a size is supplied). The per-run
resume artefacts under `models/runs/<run>/` remain alongside as the working copies; the
ensemble directory is the curated production copy.


<a id="cell-8-1"></a>

### Cell 8.1 — Production Run Selection

`SE(diff)` pools each pair's ensemble CV standard errors. No test metric participates.


In [ ]:
# Cell 8.1 — Production run: best CV ensemble RMSE; within one pooled SE -> fewer features.
phase_timer_start()   # Phase 8 starts here; Cell 8.3 prints its elapsed time.
_cvs = {lbl: run["cv"]["weighted_ensemble"] for lbl, run, _ in RUNS}
_best_label = min(_cvs, key=lambda l: _cvs[l][0])
_best_m, _best_se = _cvs[_best_label]
_candidates = [lbl for lbl, m_se in _cvs.items()
               if m_se[0] <= _best_m + float(np.hypot(_best_se, m_se[1]))]
PRODUCTION_RUN = min(
    (run for lbl, run, _ in RUNS if lbl in _candidates),
    key=lambda r: (len(r["features"]), r["cv"]["weighted_ensemble"][0]))

print("CV ensemble RMSE per registered run (development set):")
for lbl, (m, se) in _cvs.items():
    print(f"  {lbl:6s} RMSE {m:.4f} (se {se:.4f})")
print(f"Best CV: {_best_label}; runs within one pooled SE of it: {_candidates} "
      "(fewest features wins among them, better CV breaks feature-count ties)")
print(f"PRODUCTION RUN = {PRODUCTION_RUN['run_name']} "
      f"({len(PRODUCTION_RUN['features'])} features: {PRODUCTION_RUN['features']})")

<a id="cell-8-2"></a>

### Cell 8.2 — Save Artefacts

Everything the backend needs, in `models/ensemble/`: the three model files, the ensemble
weights (with the model order they belong to), the feature list, the preprocessing dictionary
(category levels and groupings, imputation medians, missing-indicator columns, TabNet's target
mean/std), and a metadata file with library versions, seed, split sizes and the final metrics.
The files are re-loaded and exercised by `predict_price_per_acre` in Cell 8.3, so a successful
run of that cell is the save/load round-trip test.


In [ ]:
# Cell 8.2 — Save the production models + every artefact the backend needs.
ENS_DIR = f"{MODEL_DIR}/ensemble"
os.makedirs(ENS_DIR, exist_ok=True)

PRODUCTION_RUN["models"]["xgb"].save_model(f"{ENS_DIR}/xgb_model.json")
PRODUCTION_RUN["models"]["catboost"].save_model(f"{ENS_DIR}/catboost_model.cbm")
PRODUCTION_RUN["models"]["tabnet"].save_model(f"{ENS_DIR}/tabnet_model")  # -> tabnet_model.zip

with open(f"{ENS_DIR}/ensemble_weights.json", "w") as f:
    json.dump({"model_order": MODEL_ORDER,
               "weights": [float(w) for w in PRODUCTION_RUN["weights"]],
               "fitted_on": "development-set out-of-fold predictions"}, f, indent=2)

with open(f"{ENS_DIR}/feature_list.json", "w") as f:
    json.dump({"run": PRODUCTION_RUN["run_name"],
               "features": PRODUCTION_RUN["features"],
               "categorical_features": [c for c in PRODUCTION_RUN["features"]
                                        if c in CATEGORICAL_CANDIDATES],
               "numeric_features": [c for c in PRODUCTION_RUN["features"]
                                    if c not in CATEGORICAL_CANDIDATES],
               "target": TARGET,
               "base_features_run_a": BASE_FEATURES}, f, indent=2)

with open(f"{ENS_DIR}/preprocessing.json", "w") as f:
    json.dump({"preprocessor": PRODUCTION_RUN["prep"].to_dict(),
               "tabnet_target": PRODUCTION_RUN["tabnet_target"],
               "train_median_log": TRAIN_MEDIAN_LOG}, f, indent=2)

_prod_test = next(_rt for _, _rn, _rt in RUNS if _rn is PRODUCTION_RUN)["metrics"][
    "weighted_ensemble"]
with open(f"{ENS_DIR}/metadata.json", "w") as f:
    json.dump({
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "production_run": PRODUCTION_RUN["run_name"],
        "seed": SEED,
        "device": DEVICE,
        "fast_debug": FAST_DEBUG,
        "library_versions": {p: _pkg_version(p) for p in
                             ["numpy", "pandas", "scipy", "scikit-learn", "xgboost",
                              "catboost", "optuna", "shap", "torch", "pytorch-tabnet"]},
        "split_sizes": {"modelling_rows": len(df_model), "development": len(dev_df),
                        "train": len(train_df), "validation": len(val_df),
                        "test": len(test_df)},
        "cv": {"n_folds": N_FOLDS, "repeats": CV_REPEATS,
               "weighted_ensemble_rmse": PRODUCTION_RUN["cv"]["weighted_ensemble"][0],
               "weighted_ensemble_se": PRODUCTION_RUN["cv"]["weighted_ensemble"][1]},
        "budgets": {"n_trials_xgb": N_TRIALS_XGB, "n_trials_catboost": N_TRIALS_CATBOOST,
                    "n_trials_tabnet": N_TRIALS_TABNET, "force_rerun": FORCE_RERUN,
                    "run_alternative": RUN_ALTERNATIVE},
        "production_run_dir": PRODUCTION_RUN["run_dir"],
        "test_metrics_weighted_ensemble": _prod_test,
    }, f, indent=2)

print(f"PHASE 8.2 — artefacts saved to {ENS_DIR}:")
for _f in ["xgb_model.json", "catboost_model.cbm", "tabnet_model.zip",
           "ensemble_weights.json", "feature_list.json", "preprocessing.json",
           "metadata.json"]:
    _p = os.path.join(ENS_DIR, _f)
    print(f"  {_f:24s} {'OK' if os.path.exists(_p) else 'MISSING!'} "
          f"({os.path.getsize(_p):,} bytes)" if os.path.exists(_p) else f"  {_f} MISSING!")

<a id="cell-8-3"></a>

### Cell 8.3 — predict_price_per_acre

The documented inference function for the backend. It loads the saved artefacts (cached after
the first call), derives `log_size_acres` from `size_acres`, the listing-date parts from
`list_date` and `has_water_body_distance` from the presence of `dist_to_water_body_km` when
needed, validates the required columns with a clear error, runs the three models, combines
their log-scale predictions with the saved weights and exponentiates.
Because `exp()` of a log-scale prediction is the **median** of the implied log-normal price
distribution, the returned price per acre is a median estimate; `total_price_ksh` multiplies
it by the parcel size.


In [ ]:
# Cell 8.3 — predict_price_per_acre: single documented entry point for the backend.
_PRODUCTION_STACK = {}

def _load_production_stack():
    """Load and cache the saved ensemble artefacts from MODEL_DIR/ensemble."""
    if not _PRODUCTION_STACK:
        with open(f"{ENS_DIR}/feature_list.json") as f:
            feature_list = json.load(f)
        with open(f"{ENS_DIR}/ensemble_weights.json") as f:
            weights = json.load(f)
        with open(f"{ENS_DIR}/preprocessing.json") as f:
            preprocessing = json.load(f)
        xgb_model = xgb.XGBRegressor()
        xgb_model.load_model(f"{ENS_DIR}/xgb_model.json")
        # load_model() restores the booster but not the sklearn-side training
        # parameters — re-apply enable_categorical so category-dtype frames keep
        # working for any consumer of this stack (including shap.TreeExplainer).
        xgb_model.set_params(enable_categorical=True, tree_method="hist")
        cat_model = CatBoostRegressor()
        cat_model.load_model(f"{ENS_DIR}/catboost_model.cbm")
        tab_model = TabNetRegressor()
        tab_model.load_model(f"{ENS_DIR}/tabnet_model.zip")
        _PRODUCTION_STACK.update({
            "features": feature_list["features"],
            "categorical": feature_list["categorical_features"],
            "model_order": weights["model_order"],
            "weights": np.asarray(weights["weights"], float),
            "prep": PipelinePreprocessor.from_dict(preprocessing["preprocessor"]),
            "tabnet_target": preprocessing["tabnet_target"],
            "models": {"xgb": xgb_model, "catboost": cat_model, "tabnet": tab_model},
        })
    return _PRODUCTION_STACK

def _prepare_input_frame(features, size_acres=None):
    """dict/DataFrame -> validated frame with the production feature columns derived."""
    stack = _load_production_stack()
    needed = list(stack["features"])
    if isinstance(features, dict):
        frame = pd.DataFrame([features])
    elif isinstance(features, pd.Series):
        frame = features.to_frame().T
    else:
        frame = features.copy()
    if size_acres is not None:
        if "size_acres" not in frame.columns:
            frame["size_acres"] = size_acres
    # derived features a caller may reasonably supply indirectly
    if "log_size_acres" in needed and "log_size_acres" not in frame.columns:
        if "size_acres" in frame.columns:
            frame["log_size_acres"] = np.log(frame["size_acres"].astype(float))
    if ("listing_year" in needed and "listing_year" not in frame.columns
            and "list_date" in frame.columns):
        _dt = pd.to_datetime(frame["list_date"], errors="coerce")
        frame["listing_year"] = _dt.dt.year.astype(float)
        frame["listing_month"] = _dt.dt.month.astype(float)
    if ("has_water_body_distance" in needed
            and "has_water_body_distance" not in frame.columns
            and "dist_to_water_body_km" in frame.columns):
        frame["has_water_body_distance"] = (frame["dist_to_water_body_km"]
                                            .notna().astype(int))
    _missing = [c for c in needed if c not in frame.columns]
    if _missing:
        raise ValueError(
            f"predict_price_per_acre: missing required feature columns: {_missing}. "
            f"Required: {needed}. Supply size_acres (log_size_acres is derived) and "
            "list_date (listing_year/listing_month are derived) where needed."
        )
    return frame[needed + (["size_acres"] if "size_acres" in frame.columns else [])]

def predict_price_per_acre(features, size_acres=None, return_details=False):
    """Predict the price per acre of a Kenyan land parcel with the saved ensemble.

    Parameters
    ----------
    features : dict, pandas.Series or pandas.DataFrame
        Raw feature values for one listing (dict or Series) or several (DataFrame).
        Keys must cover the production feature list (see feature_list.json);
        `size_acres`, `list_date` and `dist_to_water_body_km` are accepted and converted to
        `log_size_acres` / `listing_year` / `listing_month` / `has_water_body_distance`
        when those derived features are required.
    size_acres : float, optional
        Parcel size in acres; used to derive `log_size_acres` and to compute the
        total price when the input does not already carry a `size_acres` column.
    return_details : bool, default False
        Also return the individual model predictions and the log-scale ensemble value.

    Returns
    -------
    dict with 'price_per_acre_ksh' — the MEDIAN price-per-acre estimate in KSh
    (exp of the weighted mean log prediction; a median, not a mean) — and
    'total_price_ksh' = price_per_acre_ksh * size_acres whenever a size is available.
    """
    frame = _prepare_input_frame(features, size_acres=size_acres)
    stack = _load_production_stack()
    prep = stack["prep"]
    log_preds = {
        "xgb": stack["models"]["xgb"].predict(prep.transform_xgb(frame)),
        "catboost": stack["models"]["catboost"].predict(prep.transform_catboost(frame)),
        "tabnet": (stack["models"]["tabnet"]
                   .predict(prep.transform_tabnet(frame).values.astype(np.float32)).ravel()
                   * stack["tabnet_target"]["tabnet_target_std"]
                   + stack["tabnet_target"]["tabnet_target_mean"]),
    }
    w = stack["weights"]
    log_ens = sum(w[i] * log_preds[k] for i, k in enumerate(stack["model_order"]))
    ppa = np.exp(log_ens)                      # median KSh per acre

    sizes = (frame["size_acres"].astype(float).values
             if "size_acres" in frame.columns else None)
    result = {"price_per_acre_ksh": ppa}
    if sizes is not None:
        result["total_price_ksh"] = ppa * sizes
    if return_details:
        result["log_predictions"] = log_preds
        result["log_ensemble"] = log_ens
    return result

# Round-trip demo on one development row (no test data involved).
_demo_row = dev_df.iloc[[0]]
_demo_out = predict_price_per_acre(_demo_row, return_details=True)
print("PHASE 8.3 — predict_price_per_acre round-trip on development row 0:")
print(f"  price_per_acre_ksh (median): {_demo_out['price_per_acre_ksh'][0]:,.0f}")
if "total_price_ksh" in _demo_out:
    print(f"  total_price_ksh           : {_demo_out['total_price_ksh'][0]:,.0f} "
          f"(size {_demo_row['size_acres'].iloc[0]:.2f} acres)")
print(f"  actual price_per_acre_ksh  : {np.exp(_demo_row[TARGET].iloc[0]):,.0f}")
print("  (medians, not means: exp() of log-scale predictions)")
phase_timer_print("Phase 8 — save for backend")